# Modulo-10 SAE Feature Search — Qwen3-14B-IT + BFCL Tool Lists + DPE Component Analysis



In [ ]:
import os

# If authentication is needed, set HF_TOKEN outside this notebook before
# starting Python, or use `hf auth login` in a terminal.
# Hugging Face loaders use those external credentials automatically.


In [ ]:
# ============================================================
# 0. Imports, config, BFCL tool-list prompt construction, and model-native prompt rendering
# ============================================================

import os
import re
import gc
import time
import json
import math
import random
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Sequence, Union

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go

from huggingface_hub import hf_hub_download
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from sae_lens import SAE
except Exception:
    SAE = None

# -----------------------------
# Paper figure styling
# -----------------------------
PAPER_FIG_WIDTH = 900
PAPER_FIG_HEIGHT = 600
PAPER_FIG_SCALE = 4
PAPER_FONT_FAMILY = "Times New Roman"
PAPER_TITLE_FONT_SIZE = 26
PAPER_AXIS_TITLE_FONT_SIZE = 22

PAPER_FIGURE_CONFIG = {
    "toImageButtonOptions": {
        "format": "png",
        "height": PAPER_FIG_HEIGHT,
        "width": PAPER_FIG_WIDTH,
        "scale": PAPER_FIG_SCALE,
    }
}


def apply_paper_figure_style(fig):
    """Apply the uniform figure style to a Plotly figure."""
    fig.update_layout(
        width=PAPER_FIG_WIDTH,
        height=PAPER_FIG_HEIGHT,
        font=dict(family=PAPER_FONT_FAMILY),
        title=dict(font=dict(family=PAPER_FONT_FAMILY, size=PAPER_TITLE_FONT_SIZE)),
    )
    fig.update_xaxes(
        title_font=dict(family=PAPER_FONT_FAMILY, size=PAPER_AXIS_TITLE_FONT_SIZE),
        tickfont=dict(family=PAPER_FONT_FAMILY),
    )
    fig.update_yaxes(
        title_font=dict(family=PAPER_FONT_FAMILY, size=PAPER_AXIS_TITLE_FONT_SIZE),
        tickfont=dict(family=PAPER_FONT_FAMILY),
    )
    return fig


def _merge_paper_plotly_config(user_config=None):
    merged = dict(user_config or {})
    to_image = dict(merged.get("toImageButtonOptions", {}))
    to_image.update(PAPER_FIGURE_CONFIG["toImageButtonOptions"])
    merged["toImageButtonOptions"] = to_image
    return merged


if not hasattr(go.Figure, "_dpe_paper_style_original_show"):
    go.Figure._dpe_paper_style_original_show = go.Figure.show

    def _dpe_paper_style_show(self, *args, **kwargs):
        apply_paper_figure_style(self)
        user_config = kwargs.pop("config", None)
        return go.Figure._dpe_paper_style_original_show(
            self,
            *args,
            config=_merge_paper_plotly_config(user_config),
            **kwargs,
        )

    go.Figure.show = _dpe_paper_style_show



def set_seed(seed: int, verbose: bool = False):
    if verbose:
        print("seed:", seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


MODEL_NAME = 'Qwen/Qwen3-14B'
MODEL_LABEL = 'Qwen3-14B-IT'
USE_CHAT_TEMPLATE = True
ADD_BOS_TOKEN = False
CHAT_TEMPLATE_KWARGS = {'enable_thinking': False}
CHAT_TEMPLATE_USE_SYSTEM_ROLE = True
PROMPT_FORMAT_NAME = 'qwen3_chat_template_system_plus_user_bfcl_tool_list_enable_thinking_false'

SYSTEM_PROMPT = 'You are a careful tool-use assistant. You will read an ordered catalog of available tools/functions. Track the sequence of tool definitions carefully.'
AGNEWS_SYSTEM_PROMPT = SYSTEM_PROMPT

TARGET_LAYERS = [10, 20, 30]

SAE_PROVIDER = 'qwen_batchtopk_direct'
SAE_RELEASE = None
SAE_ID_TEMPLATE = None

LLAMA_SCOPE_REPO_ID = None
LLAMA_SCOPE_LAYER_DIR_TEMPLATE = None
LLAMA_SCOPE_CHECKPOINT_FILENAME = "checkpoints/final.safetensors"
LLAMA_SCOPE_HYPERPARAMS_FILENAME = "hyperparams.json"

QWEN_SAE_REPO_ID = 'adamkarvonen/qwen3-14b-saes'
QWEN_SAE_FILENAME_TEMPLATE = 'saes_Qwen_Qwen3-14B_batch_top_k/resid_post_layer_{layer}/trainer_{trainer}/ae.pt'
QWEN_SAE_TRAINER = 0
QWEN_FEATURE_VALUE_MODE = 'preactivation'
SAELENS_FEATURE_VALUE_MODE = 'preactivation'
GEMMA_SCOPE_FEATURE_VALUE_MODE = SAELENS_FEATURE_VALUE_MODE
LLAMA_SCOPE_FEATURE_VALUE_MODE = SAELENS_FEATURE_VALUE_MODE

# BFCL tool-list dataset.
TASK_TYPE = "bfcl_tool"
ITEM_WORD = "Tool"
ITEM_WORD_LOWER = "tool"
ITEM_WORD_PLURAL = "tools"
ITEM_MARKER_WORD = ITEM_WORD
ITEM_MARKER_WORD_LOWER = ITEM_WORD_LOWER
ITEM_MARKER_WORD_PLURAL = ITEM_WORD_PLURAL
N_TOOLS = 51                    # Tool 1 ... Tool 51, giving completed counts 1 ... 50 before Tool 2 ... 51.
N_SHOTS = N_TOOLS               # Backward-compatible alias used by generic downstream metadata.
NUM_PROMPTS = 300
EXPECTED_NUM_EXAMPLES = N_TOOLS

BFCL_REPO_ID = "gorilla-llm/Berkeley-Function-Calling-Leaderboard"
BFCL_DATA_FILE = "BFCL_v3_multiple.json"
MAX_TOOL_DESCRIPTION_CHARS = 240
MAX_PARAM_DESCRIPTION_CHARS = 120
MAX_PARAMS_PER_TOOL = 8

MODULO_TARGET_COMPLETED_COUNTS = [10, 20, 30, 40, 50]
TARGET_MARKER_NUMBERS = [c + 1 for c in MODULO_TARGET_COMPLETED_COUNTS]

positions = [-1] # [-1], "all"
ALL_EXEMPLAR_POSITION_MODE = "all"
SAE_ENCODE_CHUNK_SIZE = 2048


def is_all_position_selection(position_spec) -> bool:
    return isinstance(position_spec, str) and position_spec.strip().lower() == ALL_EXEMPLAR_POSITION_MODE


def normalize_position_offsets(position_spec):
    if is_all_position_selection(position_spec):
        return ALL_EXEMPLAR_POSITION_MODE
    if isinstance(position_spec, str):
        raise ValueError("positions must be either 'all' or a non-empty iterable of negative integer offsets, e.g. [-1].")
    offsets = [int(p) for p in position_spec]
    if len(offsets) == 0:
        raise ValueError("positions must contain at least one token offset, e.g. [-1], or be set to 'all'.")
    if any(p >= 0 for p in offsets):
        raise ValueError("positions must contain negative offsets only: -1 is right before Tool, -2 is two tokens before Tool, etc.")
    if len(set(offsets)) != len(offsets):
        raise ValueError(f"positions contains duplicate offsets: {offsets}")
    return offsets


POSITION_OFFSETS = normalize_position_offsets(positions)


def position_selection_to_jsonable(position_spec):
    normalized = normalize_position_offsets(position_spec)
    if is_all_position_selection(normalized):
        return ALL_EXEMPLAR_POSITION_MODE
    return [int(p) for p in normalized]


def position_selection_label(position_spec) -> str:
    normalized = normalize_position_offsets(position_spec)
    if is_all_position_selection(normalized):
        return "all tokens in previous completed tool block, excluding the Tool header"
    return ", ".join(str(p) for p in normalized)


def _format_position_offsets_for_path(position_offsets):
    position_offsets = normalize_position_offsets(position_offsets)
    if is_all_position_selection(position_offsets):
        return "positions_all_previous_tool_block"
    parts = []
    for p in position_offsets:
        p = int(p)
        if p < 0:
            parts.append(f"m{abs(p)}")
        elif p > 0:
            parts.append(f"p{p}")
        else:
            parts.append("0")
    return "positions_" + "_".join(parts)


POSITIONS_TAG = _format_position_offsets_for_path(POSITION_OFFSETS)
MAX_PROMPTS = None
VERBOSE_EVERY = 25
EMPTY_CACHE_EVERY = 50
SAE_VALUE_NAME = 'preactivation'
SAE_VALUE_LABEL = 'SAE feature pre-activation'
TOP_K_FOR_LAYER_METRIC = 20
LAYER_METRIC_TOP_K_OPTIONS = [10, 20, 50]
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
SAE_DTYPE = torch.float32
OUTPUT_DIR = f"./{MODEL_LABEL}_{SAE_PROVIDER}_bfcl_tool_modulo10_single_forward_layer_sweep_outputs_{POSITIONS_TAG}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
set_seed(42)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

# -----------------------------
# Figure styling
# -----------------------------

FIG_WIDTH = 900
FIG_HEIGHT = 600
FIG_EXPORT_SCALE = 4
FIG_FONT_FAMILY = "Times New Roman"
FIG_TITLE_FONT_SIZE = 26
FIG_AXIS_TITLE_FONT_SIZE = 22


def plotly_export_config(filename: str = "dpe_figure"):
    return {
        "toImageButtonOptions": {
            "format": "png",
            "filename": filename,
            "height": FIG_HEIGHT,
            "width": FIG_WIDTH,
            "scale": FIG_EXPORT_SCALE,
        }
    }


def apply_plotly_style(fig):
    """Apply paper/export styling to every Plotly figure."""
    fig.update_layout(
        width=FIG_WIDTH,
        height=FIG_HEIGHT,
        font=dict(family=FIG_FONT_FAMILY),
        title=dict(font=dict(family=FIG_FONT_FAMILY, size=FIG_TITLE_FONT_SIZE)),
        legend=dict(font=dict(family=FIG_FONT_FAMILY)),
        template="seaborn",
    )
    fig.update_xaxes(
        title_font=dict(family=FIG_FONT_FAMILY, size=FIG_AXIS_TITLE_FONT_SIZE),
        tickfont=dict(family=FIG_FONT_FAMILY),
    )
    fig.update_yaxes(
        title_font=dict(family=FIG_FONT_FAMILY, size=FIG_AXIS_TITLE_FONT_SIZE),
        tickfont=dict(family=FIG_FONT_FAMILY),
    )
    return fig


def show_figure(fig, filename: str = "dpe_figure"):
    apply_plotly_style(fig)
    fig.show(config=plotly_export_config(filename))
    return fig

print("MODEL_NAME:", MODEL_NAME)
print("TASK_TYPE:", TASK_TYPE)
print("ITEM_WORD:", ITEM_WORD)
print("TARGET_LAYERS:", TARGET_LAYERS)
print("EXPECTED_NUM_EXAMPLES:", EXPECTED_NUM_EXAMPLES)
print("TARGET_MARKER_NUMBERS:", TARGET_MARKER_NUMBERS)
print("POSITION_OFFSETS / POSITION_SELECTION:", POSITION_OFFSETS)
print("OUTPUT_DIR:", OUTPUT_DIR)

# -----------------------------
# BFCL tool prompt helpers
# -----------------------------

def load_json_or_jsonl_from_hf(repo_id: str, filename: str, repo_type: str = "dataset"):
    """Download and parse a JSON or JSONL file from Hugging Face without relying on dataset scripts."""
    path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type=repo_type)
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            for key in ["data", "examples", "rows"]:
                if key in obj and isinstance(obj[key], list):
                    return obj[key]
            return [obj]
    except json.JSONDecodeError:
        pass

    rows = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        s = line.strip()
        if not s:
            continue
        try:
            rows.append(json.loads(s))
        except json.JSONDecodeError as e:
            raise ValueError(f"Failed to parse {filename} line {line_no}: {e}\n{s[:500]}")
    return rows


def normalize_text(x: Any, max_chars: Optional[int] = None) -> str:
    s = str(x).replace("\n", " ").replace("\t", " ").strip()
    s = re.sub(r"\s+", " ", s)
    if max_chars is not None and len(s) > max_chars:
        # Do not append an artificial ellipsis before every Tool marker.
        truncated = s[:max_chars].rstrip()
        last_space = truncated.rfind(" ")
        if last_space >= int(0.55 * max_chars):
            truncated = truncated[:last_space].rstrip()
        s = truncated
    return s


def _canonical_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except TypeError:
        return str(x)


def compact_tool_schema(func: Dict[str, Any], idx: int) -> str:
    """Compact one-function definition. Every block begins with "Tool k:"."""
    name = normalize_text(func.get("name", f"tool_{idx}"))
    desc = normalize_text(func.get("description", ""), max_chars=MAX_TOOL_DESCRIPTION_CHARS)
    params = func.get("parameters", {}) or {}
    props = params.get("properties", {}) if isinstance(params, dict) else {}
    required = params.get("required", []) if isinstance(params, dict) else []

    param_parts = []
    if isinstance(props, dict):
        for p_name, p_info in list(props.items())[:MAX_PARAMS_PER_TOOL]:
            if not isinstance(p_info, dict):
                p_type = "unknown"
                p_desc = ""
            else:
                p_type = p_info.get("type", "unknown")
                p_desc = normalize_text(p_info.get("description", ""), max_chars=MAX_PARAM_DESCRIPTION_CHARS)
            req_mark = "required" if p_name in required else "optional"
            if p_desc:
                param_parts.append(f"{p_name}: {p_type} ({req_mark}; {p_desc})")
            else:
                param_parts.append(f"{p_name}: {p_type} ({req_mark})")

    params_str = "; ".join(param_parts) if param_parts else "none"
    return f"Tool {idx}:\nName: {name}\nDescription: {desc}\nParameters: {params_str}"


def build_tool_pool():
    """Load BFCL and flatten all function definitions into a deduplicated tool pool."""
    print("Loading BFCL tool-use data:", BFCL_REPO_ID, BFCL_DATA_FILE)
    data_rows = load_json_or_jsonl_from_hf(BFCL_REPO_ID, BFCL_DATA_FILE, repo_type="dataset")

    tool_pool = []
    seen = set()
    for row_idx, row in enumerate(data_rows):
        funcs = row.get("function", []) or []
        if not isinstance(funcs, list):
            continue
        for func_idx, func in enumerate(funcs):
            if not isinstance(func, dict):
                continue
            name = normalize_text(func.get("name", f"tool_{row_idx}_{func_idx}"))
            if len(name) == 0:
                continue
            sig = _canonical_json({
                "name": func.get("name", ""),
                "description": func.get("description", ""),
                "parameters": func.get("parameters", {}),
            })
            if sig in seen:
                continue
            seen.add(sig)
            tool_pool.append({
                "tool_idx": len(tool_pool),
                "source_row_idx": row_idx,
                "source_func_idx": func_idx,
                "name": name,
                "schema": func,
                "signature": sig,
            })

    if len(tool_pool) < N_TOOLS:
        raise ValueError(f"Need at least {N_TOOLS} unique tools, found {len(tool_pool)}")

    print(f"Loaded {len(tool_pool)} unique individual tools from BFCL.")
    print("Example compact tool schema:")
    print(compact_tool_schema(tool_pool[0]["schema"], 1)[:1200])
    return tool_pool


def render_tool_prompt_body(tools: List[Dict[str, Any]]) -> str:
    header = (
        "You are given a catalog of available tools/functions. Each numbered item is one tool definition.\n"
        "Study the list of tools and their parameter schemas. Do not answer; just read the sequence of tool definitions.\n\n"
        "Available tool definitions:\n"
    )
    blocks = [compact_tool_schema(t["schema"], i + 1) for i, t in enumerate(tools)]
    return header + "\n\n".join(blocks) + "\n"


def build_prompt_sets(tokenizer, num_prompts: int = NUM_PROMPTS, n_items: int = N_TOOLS):
    """Build random BFCL tool-list prompts."""
    tool_pool = build_tool_pool()
    prompts = []
    prompt_bodies = []
    meta = []
    for prompt_idx in range(num_prompts):
        local_rng = random.Random(100_000 + prompt_idx)
        sampled_indices = local_rng.sample(range(len(tool_pool)), n_items)
        tools = [tool_pool[i] for i in sampled_indices]
        body = render_tool_prompt_body(tools)
        prompt = render_model_prompt(tokenizer=tokenizer, user_content=body, system_content=SYSTEM_PROMPT)
        prompts.append(prompt)
        prompt_bodies.append(body)
        meta.append({
            "prompt_idx": prompt_idx,
            "n_tools": n_items,
            "tool_indices": sampled_indices,
            "tool_names": [tool_pool[i]["name"] for i in sampled_indices],
        })
    return prompts, meta, prompt_bodies

def ensure_single_bos_prefix(text: str, tokenizer) -> str:
    if not ADD_BOS_TOKEN:
        return text
    bos = getattr(tokenizer, "bos_token", None)
    if not isinstance(bos, str) or len(bos) == 0:
        return text
    if text.startswith(bos):
        return text
    return bos + text


def _manual_qwen_chatml_render(system_content: str, user_content: str) -> str:
    rendered = (
        f"<|im_start|>system\n{system_content}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    if CHAT_TEMPLATE_KWARGS.get("enable_thinking") is False:
        rendered += "<think>\n\n</think>\n"
    return rendered


def render_model_prompt(tokenizer, user_content: str, system_content: Optional[str] = None):
    system_content = system_content if system_content is not None else SYSTEM_PROMPT
    if not USE_CHAT_TEMPLATE:
        plain = f"{system_content}\n\n{user_content}" if system_content else user_content
        return ensure_single_bos_prefix(plain, tokenizer)
    rendered_user_content = user_content
    if system_content and not CHAT_TEMPLATE_USE_SYSTEM_ROLE:
        rendered_user_content = f"{system_content}\n\n{user_content}"
    messages = []
    if CHAT_TEMPLATE_USE_SYSTEM_ROLE and system_content:
        messages.append({"role": "system", "content": system_content})
    messages.append({"role": "user", "content": rendered_user_content})
    try:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_TEMPLATE_KWARGS)
        return ensure_single_bos_prefix(rendered, tokenizer)
    except TypeError:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return ensure_single_bos_prefix(rendered, tokenizer)
    except Exception as e:
        print("Warning: tokenizer.apply_chat_template failed. Falling back to a simple rendered prompt.")
        print("Reason:", repr(e))
        if CHAT_TEMPLATE_USE_SYSTEM_ROLE and system_content and "qwen" in MODEL_NAME.lower():
            return ensure_single_bos_prefix(_manual_qwen_chatml_render(system_content, user_content), tokenizer)
        return ensure_single_bos_prefix(rendered_user_content, tokenizer)


def sanity_check_prompt(prompt: str):
    print("=" * 80)
    print(f"{MODEL_LABEL} BFCL TOOL-LIST PROMPT SANITY CHECK")
    print("=" * 80)
    print("Prompt starts with:")
    print(repr(prompt[:500]))
    pattern = rf"(?m)^{re.escape(ITEM_WORD)}\s+\d+:?"
    print("First marker occurrence:", prompt.find(ITEM_WORD))
    print("Num anchored headers:", len(re.findall(pattern, prompt)))
    print("Expected headers:", EXPECTED_NUM_EXAMPLES)
    artificial_pattern = r"\s\.\.\.\s*\n\n" + re.escape(ITEM_WORD) + r"\s+\d+:?"
    print("Contains artificial ellipsis before marker:", len(re.findall(artificial_pattern, prompt)))
    print("=" * 80)


In [ ]:

# ============================================================
# 1. Load model, tokenizer, and BFCL few-shot prompts
# ============================================================

torch.backends.cuda.matmul.allow_tf32 = True

if not torch.cuda.is_available():
    print("Warning: torch.cuda.is_available() is False. This model will likely be impractical on CPU.")

print("Loading tokenizer:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
)

if not getattr(tokenizer, "is_fast", False):
    raise ValueError("A fast tokenizer is required because this notebook uses return_offsets_mapping=True.")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Building BFCL tool-list prompts...")
original_prompts, original_meta, original_prompt_bodies = build_prompt_sets(tokenizer=tokenizer)

print(f"BFCL tool prompts: {len(original_prompts)}")
print(f"N_TOOLS: {N_TOOLS}")
print(f"EXPECTED_NUM_EXAMPLES / headers: {EXPECTED_NUM_EXAMPLES}")
sanity_check_prompt(original_prompts[0])

print("Loading model:", MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=DTYPE,
    trust_remote_code=True,
)
model.eval()


def get_input_device(model):
    return model.get_input_embeddings().weight.device


def get_module_device(module):
    try:
        return next(module.parameters()).device
    except StopIteration:
        try:
            return next(module.buffers()).device
        except StopIteration:
            return get_input_device(model)


model_device = get_input_device(model)
print("MODEL_NAME:", MODEL_NAME)
print("model.config.num_hidden_layers:", model.config.num_hidden_layers)
print("model input device:", model_device)
print("tokenizer.eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
print("tokenizer.pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("tokenizer.bos_token:", repr(getattr(tokenizer, "bos_token", None)), getattr(tokenizer, "bos_token_id", None))
print("ADD_BOS_TOKEN:", ADD_BOS_TOKEN)

for layer in TARGET_LAYERS:
    if layer >= int(model.config.num_hidden_layers):
        raise ValueError(
            f"Requested layer {layer}, but model.config.num_hidden_layers={model.config.num_hidden_layers}. "
            "Layer indices are zero-indexed."
        )
    layer_device = get_module_device(model.get_submodule(f"model.layers.{layer}"))
    print(f"Layer {layer} device:", layer_device)


In [ ]:
print(original_prompts[0])

In [ ]:
# ============================================================
# 2. SAE loading, token utilities, and single-forward multi-layer capture
# ============================================================

EXAMPLE_MARKER_RE = re.compile(rf"(?m)^{re.escape(ITEM_WORD)}\s+(\d+):?")
# Backward-compatible name: downstream cells still refer to "example" marker helpers.


def get_backbone_model(model):
    # Most HF causal-LM wrappers expose the transformer body as `.model`.
    return getattr(model, "model", model)


def get_sae_device(sae, fallback=None):
    for p in sae.parameters():
        return p.device
    for b in sae.buffers():
        return b.device
    return fallback if fallback is not None else get_input_device(model)


def get_sae_dtype(sae, fallback=torch.float32):
    for p in sae.parameters():
        return p.dtype
    for b in sae.buffers():
        if b.dtype.is_floating_point:
            return b.dtype
    return fallback


def move_sae_to_device_and_dtype(sae, device, dtype=SAE_DTYPE):
    try:
        sae = sae.to(device=device, dtype=dtype)
    except TypeError:
        sae = sae.to(device=device)
        try:
            sae = sae.to(dtype=dtype)
        except Exception:
            pass
    return sae


def describe_sae_for_layer(layer: int) -> Dict[str, Any]:
    layer = int(layer)
    if SAE_PROVIDER in {"saelens", "gemma_scope_saelens"}:
        return {
            "provider": SAE_PROVIDER,
            "release": SAE_RELEASE,
            "sae_id": SAE_ID_TEMPLATE.format(layer=layer),
        }
    if SAE_PROVIDER == "llama_scope_direct":
        layer_dir = LLAMA_SCOPE_LAYER_DIR_TEMPLATE.format(layer=layer)
        return {
            "provider": SAE_PROVIDER,
            "repo_id": LLAMA_SCOPE_REPO_ID,
            "layer_dir": layer_dir,
            "checkpoint": f"{layer_dir}/{LLAMA_SCOPE_CHECKPOINT_FILENAME}",
            "hyperparams": f"{layer_dir}/{LLAMA_SCOPE_HYPERPARAMS_FILENAME}",
        }
    if SAE_PROVIDER == "qwen_batchtopk_direct":
        filename = QWEN_SAE_FILENAME_TEMPLATE.format(layer=layer, trainer=QWEN_SAE_TRAINER)
        return {
            "provider": SAE_PROVIDER,
            "repo_id": QWEN_SAE_REPO_ID,
            "filename": filename,
            "trainer": QWEN_SAE_TRAINER,
            "feature_value_mode": QWEN_FEATURE_VALUE_MODE,
        }
    raise ValueError(f"Unknown SAE_PROVIDER: {SAE_PROVIDER!r}")


# -----------------------------
# SAE via SAE Lens
# -----------------------------

def load_gemma_scope_sae_for_layer(layer: int):
    if SAE is None:
        raise ImportError(
            "SAE_PROVIDER='gemma_scope_saelens' requires sae-lens. "
            "Install it in this kernel, e.g. `python -m pip install -U sae-lens`."
        )
    layer = int(layer)
    target_layer_device = get_module_device(model.get_submodule(f"model.layers.{layer}"))
    sae_id = SAE_ID_TEMPLATE.format(layer=layer)
    print("=" * 100)
    print(f"Loading SAE Lens SAE for layer {layer}")
    print("release:", SAE_RELEASE)
    print("sae_id:", sae_id)
    print("target_layer_device:", target_layer_device)

    try:
        loaded = SAE.from_pretrained(
            release=SAE_RELEASE,
            sae_id=sae_id,
            device=str(target_layer_device),
        )
    except TypeError:
        loaded = SAE.from_pretrained(
            release=SAE_RELEASE,
            sae_id=sae_id,
        )

    if isinstance(loaded, tuple):
        sae = loaded[0]
        cfg_dict = loaded[1] if len(loaded) > 1 else None
        sparsity = loaded[2] if len(loaded) > 2 else None
    else:
        sae = loaded
        cfg_dict = None
        sparsity = None

    sae = move_sae_to_device_and_dtype(sae, device=target_layer_device, dtype=SAE_DTYPE)
    sae.eval()

    d_in = getattr(getattr(sae, "cfg", None), "d_in", None)
    d_sae = getattr(getattr(sae, "cfg", None), "d_sae", None)
    print("Loaded SAE cfg d_in:", d_in)
    print("Loaded SAE cfg d_sae:", d_sae)
    print("SAE device:", get_sae_device(sae, fallback=target_layer_device))
    print("SAE dtype:", get_sae_dtype(sae))
    return sae


# -----------------------------
# Qwen3 BatchTopK direct wrapper
# -----------------------------

class QwenBatchTopKSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Adam Karvonen's Qwen3 BatchTopK SAEs.

    The model card says these BatchTopK SAEs have a fixed scalar threshold and can be used
    as JumpReLU-style SAEs at inference. This wrapper exposes `encode` returning post-threshold
    feature activations by default; set QWEN_FEATURE_VALUE_MODE="preactivation" to analyze preacts.
    """

    def __init__(self, state_dict, device, dtype):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        if isinstance(state_dict, dict) and "model" in state_dict and isinstance(state_dict["model"], dict):
            state_dict = state_dict["model"]

        b_dec = state_dict["b_dec"].detach()
        b_enc = state_dict.get("encoder.bias", state_dict.get("b_enc")).detach()
        enc_w = state_dict.get("encoder.weight", state_dict.get("W_enc")).detach()
        dec_w = state_dict.get("decoder.weight", state_dict.get("W_dec")).detach()
        threshold = state_dict["threshold"]
        threshold = threshold.detach() if torch.is_tensor(threshold) else torch.tensor(threshold)
        k = state_dict.get("k", None)
        if torch.is_tensor(k):
            k = int(k.item())

        d_model = int(b_dec.numel())
        n_features = int(b_enc.numel())

        if enc_w.shape == (n_features, d_model):
            W_enc = enc_w.T.contiguous()
        elif enc_w.shape == (d_model, n_features):
            W_enc = enc_w.contiguous()
        else:
            raise ValueError(f"Unexpected encoder.weight shape {tuple(enc_w.shape)}")

        if dec_w.shape == (d_model, n_features):
            W_dec = dec_w.T.contiguous()
        elif dec_w.shape == (n_features, d_model):
            W_dec = dec_w.contiguous()
        else:
            raise ValueError(f"Unexpected decoder.weight shape {tuple(dec_w.shape)}")

        self.register_buffer("W_enc", W_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.to(device=self.device, dtype=self.dtype))

        self.k = k
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.b_dec

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, pre, torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        pre = self.encode_pre(x)
        if QWEN_FEATURE_VALUE_MODE == "preactivation":
            return pre
        if QWEN_FEATURE_VALUE_MODE != "activation":
            raise ValueError(f"Unknown QWEN_FEATURE_VALUE_MODE={QWEN_FEATURE_VALUE_MODE!r}")
        return self.activation_fn(pre)


def load_qwen_batchtopk_sae_for_layer(layer: int):
    layer = int(layer)
    target_layer_device = get_module_device(model.get_submodule(f"model.layers.{layer}"))
    filename = QWEN_SAE_FILENAME_TEMPLATE.format(layer=layer, trainer=QWEN_SAE_TRAINER)
    print("=" * 100)
    print(f"Loading Qwen3 BatchTopK SAE for layer {layer}")
    print("repo_id:", QWEN_SAE_REPO_ID)
    print("filename:", filename)
    print("target_layer_device:", target_layer_device)
    path = hf_hub_download(repo_id=QWEN_SAE_REPO_ID, filename=filename)
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    sae = QwenBatchTopKSAE(state, device=target_layer_device, dtype=SAE_DTYPE)
    sae.eval()
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"k={sae.k}, threshold_mean={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


# -----------------------------
# Llama Scope direct safetensors wrapper
# -----------------------------

class LlamaScopeSAE(torch.nn.Module):
    """
    Minimal inference wrapper for OpenMOSS Llama Scope residual-stream SAEs.

    This follows the Llamascopium SparseAutoEncoder encode path:
      hidden_pre = x @ W_E + b_E
      if sparsity_include_decoder_norm: hidden_pre *= ||W_D||
      feature_acts = JumpReLU(hidden_pre)
      if sparsity_include_decoder_norm: feature_acts /= ||W_D||

    For checkpoints with norm_activation='dataset-wise', this wrapper applies the same inference-time
    parameter standardization implemented in Llamascopium's `standardize_parameters_of_dataset_norm`.
    """

    def __init__(self, state_dict: Dict[str, torch.Tensor], hyperparams: Dict[str, Any], device, dtype):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype
        self.hyperparams = dict(hyperparams)

        self.d_model = int(hyperparams.get("d_model"))
        self.d_sae = int(hyperparams.get("d_sae"))
        self.n_features = self.d_sae
        self.act_fn = str(hyperparams.get("act_fn", "jumprelu")).lower()
        self.sparsity_include_decoder_norm = bool(hyperparams.get("sparsity_include_decoder_norm", False))
        self.use_decoder_bias = bool(hyperparams.get("use_decoder_bias", True))
        self.hook_point_in = hyperparams.get("hook_point_in", "in")
        self.hook_point_out = hyperparams.get("hook_point_out", self.hook_point_in)
        self.top_k = int(hyperparams.get("top_k", 0) or 0)

        W_E = self._get_required_tensor(state_dict, ["W_E", "encoder.weight", "W_enc"], shape_options=[(self.d_model, self.d_sae), (self.d_sae, self.d_model)])
        if W_E.shape == (self.d_sae, self.d_model):
            W_E = W_E.T.contiguous()
        W_D = self._get_required_tensor(state_dict, ["W_D", "decoder.weight", "W_dec"], shape_options=[(self.d_sae, self.d_model), (self.d_model, self.d_sae)])
        if W_D.shape == (self.d_model, self.d_sae):
            W_D = W_D.T.contiguous()
        b_E = self._get_optional_tensor(state_dict, ["b_E", "encoder.bias", "b_enc"], shape=(self.d_sae,))
        if b_E is None:
            b_E = torch.zeros(self.d_sae)
        b_D = self._get_optional_tensor(state_dict, ["b_D", "decoder.bias", "b_dec"], shape=(self.d_model,))
        if b_D is None:
            b_D = torch.zeros(self.d_model)

        threshold = self._get_optional_tensor(
            state_dict,
            ["activation_function.log_jumprelu_threshold", "log_jumprelu_threshold"],
            shape=(self.d_sae,),
        )
        if threshold is not None:
            threshold = threshold.detach().float().exp()
        else:
            threshold = self._get_optional_tensor(
                state_dict,
                ["activation_function.jumprelu_threshold", "jumprelu_threshold", "threshold"],
                shape=(self.d_sae,),
            )
        if threshold is None:
            threshold_value = float(hyperparams.get("jump_relu_threshold", 0.0))
            threshold = torch.full((self.d_sae,), threshold_value)

        # Llamascopium inference-time standardization for dataset-wise activation normalization.
        if hyperparams.get("norm_activation", None) == "dataset-wise" and hyperparams.get("dataset_average_activation_norm") is not None:
            avg = hyperparams["dataset_average_activation_norm"]
            norm_in = float(avg.get(self.hook_point_in, avg.get("in")))
            norm_out = float(avg.get(self.hook_point_out, avg.get("out", norm_in)))
            input_norm_factor = math.sqrt(self.d_model) / norm_in
            output_norm_factor = math.sqrt(self.d_model) / norm_out
            b_E = b_E / input_norm_factor
            b_D = b_D / output_norm_factor
            W_D = W_D * (input_norm_factor / output_norm_factor)
            self.norm_activation_standardized = True
        else:
            self.norm_activation_standardized = False

        self.register_buffer("W_E", W_E.to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_D", W_D.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_E", b_E.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_D", b_D.to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.to(device=self.device, dtype=self.dtype))

    @staticmethod
    def _get_optional_tensor(state_dict, candidate_keys, shape=None):
        for key in candidate_keys:
            if key in state_dict:
                tensor = state_dict[key].detach().cpu()
                if shape is None or tuple(tensor.shape) == tuple(shape):
                    return tensor
        if shape is not None:
            matches = []
            for key, tensor in state_dict.items():
                if torch.is_tensor(tensor) and tuple(tensor.shape) == tuple(shape):
                    lowered = key.lower()
                    if any(c.lower() in lowered for c in candidate_keys):
                        matches.append((key, tensor.detach().cpu()))
            if len(matches) == 1:
                print(f"Using shape/key fuzzy match for {candidate_keys}: {matches[0][0]}")
                return matches[0][1]
        return None

    @staticmethod
    def _get_required_tensor(state_dict, candidate_keys, shape_options):
        for key in candidate_keys:
            if key in state_dict:
                tensor = state_dict[key].detach().cpu()
                if any(tuple(tensor.shape) == tuple(s) for s in shape_options):
                    return tensor
        for key, tensor in state_dict.items():
            if not torch.is_tensor(tensor):
                continue
            if any(tuple(tensor.shape) == tuple(s) for s in shape_options):
                lowered = key.lower()
                if any(c.lower() in lowered for c in candidate_keys):
                    print(f"Using shape/key fuzzy match for {candidate_keys}: {key}")
                    return tensor.detach().cpu()
        available = {k: tuple(v.shape) for k, v in state_dict.items() if torch.is_tensor(v)}
        raise KeyError(f"Could not find required tensor {candidate_keys} with shape_options={shape_options}. Available tensor shapes: {available}")

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        hidden_pre = x @ self.W_E + self.b_E
        decoder_norm = None
        if self.sparsity_include_decoder_norm:
            decoder_norm = torch.norm(self.W_D, p=2, dim=1).clamp_min(1e-8)
            hidden_pre = hidden_pre * decoder_norm
        if self.act_fn == "jumprelu":
            feature_acts = torch.where(hidden_pre > self.threshold, hidden_pre, torch.zeros_like(hidden_pre))
        elif self.act_fn == "relu":
            feature_acts = torch.relu(hidden_pre)
        elif self.act_fn == "topk":
            k = int(self.top_k)
            if k <= 0:
                raise ValueError("LlamaScopeSAE act_fn='topk' but top_k is not positive.")
            vals, idx = torch.topk(hidden_pre, k=min(k, hidden_pre.shape[-1]), dim=-1)
            vals = torch.relu(vals)
            feature_acts = torch.zeros_like(hidden_pre)
            feature_acts.scatter_(dim=-1, index=idx, src=vals)
        else:
            raise ValueError(f"Unsupported Llama Scope act_fn={self.act_fn!r}")
        if self.sparsity_include_decoder_norm:
            feature_acts = feature_acts / decoder_norm
        return feature_acts


def load_llama_scope_sae_for_layer(layer: int):
    layer = int(layer)
    target_layer_device = get_module_device(model.get_submodule(f"model.layers.{layer}"))
    layer_dir = LLAMA_SCOPE_LAYER_DIR_TEMPLATE.format(layer=layer)
    checkpoint_filename = f"{layer_dir}/{LLAMA_SCOPE_CHECKPOINT_FILENAME}"
    hyperparams_filename = f"{layer_dir}/{LLAMA_SCOPE_HYPERPARAMS_FILENAME}"
    print("=" * 100)
    print(f"Loading Llama Scope SAE for layer {layer}")
    print("repo_id:", LLAMA_SCOPE_REPO_ID)
    print("checkpoint:", checkpoint_filename)
    print("hyperparams:", hyperparams_filename)
    print("target_layer_device:", target_layer_device)

    try:
        from safetensors.torch import load_file as load_safetensors_file
    except Exception as e:
        raise ImportError("Llama Scope direct loading requires safetensors: `python -m pip install safetensors`.") from e

    ckpt_path = hf_hub_download(repo_id=LLAMA_SCOPE_REPO_ID, filename=checkpoint_filename)
    hyperparams_path = hf_hub_download(repo_id=LLAMA_SCOPE_REPO_ID, filename=hyperparams_filename)
    with open(hyperparams_path, "r", encoding="utf-8") as f:
        hyperparams = json.load(f)
    state = load_safetensors_file(ckpt_path, device="cpu")
    print("State tensor keys:", sorted(state.keys()))
    sae = LlamaScopeSAE(state, hyperparams=hyperparams, device=target_layer_device, dtype=SAE_DTYPE)
    sae.eval()
    print(
        f"Loaded Llama Scope SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"act_fn={sae.act_fn}, dataset_norm_standardized={sae.norm_activation_standardized}"
    )
    return sae


def load_sae_for_layer(layer: int):
    if SAE_PROVIDER in {"saelens", "gemma_scope_saelens"}:
        return load_gemma_scope_sae_for_layer(layer)
    if SAE_PROVIDER == "llama_scope_direct":
        return load_llama_scope_sae_for_layer(layer)
    if SAE_PROVIDER == "qwen_batchtopk_direct":
        return load_qwen_batchtopk_sae_for_layer(layer)
    raise ValueError(f"Unknown SAE_PROVIDER: {SAE_PROVIDER!r}")


def load_all_saes_for_layers(layers=TARGET_LAYERS):
    """Load all requested SAEs once so that each prompt only needs one model forward pass."""
    saes_by_layer = {}
    for layer in layers:
        layer = int(layer)
        saes_by_layer[layer] = load_sae_for_layer(layer)
    return saes_by_layer


def find_token_overlapping_char(offset_mapping, char_pos):
    """
    Return token index whose character span overlaps char_pos.
    If none overlaps, return the first non-special token starting after char_pos.
    """
    fallback = None

    for tok_idx, (s, e) in enumerate(offset_mapping):
        if s == 0 and e == 0:
            continue

        if s <= char_pos < e:
            return tok_idx

        if fallback is None and s > char_pos:
            fallback = tok_idx

    return fallback


def get_example_marker_prediction_positions(prompt, input_ids_cpu, offset_mapping):
    """
    For each anchored item marker, find the causal-LM position immediately before
    the marker's first token. The function name is kept for backward compatibility.
    """
    results = []
    for m in EXAMPLE_MARKER_RE.finditer(prompt):
        item_char_start = m.start()
        item_tok = find_token_overlapping_char(offset_mapping, item_char_start)
        if item_tok is None or item_tok <= 0:
            continue
        pred_pos = item_tok - 1
        target_id = int(input_ids_cpu[item_tok])
        target_decoded = tokenizer.decode([target_id])
        marker_number = int(m.group(1))
        results.append({
            "pred_pos": int(pred_pos),
            "target_id": target_id,
            "target_decoded": target_decoded,
            "example_token_pos": int(item_tok),
            "item_token_pos": int(item_tok),
            "char_start": int(item_char_start),
            "char_end": int(m.end()),
            "match": m.group(0),
            "marker_number": marker_number,
            "completed_examples": marker_number - 1,
        })
    marker_numbers_found = [x["marker_number"] for x in results]
    expected = list(range(1, EXPECTED_NUM_EXAMPLES + 1))
    if marker_numbers_found != expected:
        raise ValueError(
            f"{ITEM_WORD} marker detection failed or prompt is malformed.\n"
            f"Expected marker sequence: {expected}\n"
            f"Found marker sequence: {marker_numbers_found[:80]}"
        )
    for idx, info in enumerate(results):
        info["previous_exemplar_marker_number"] = None
        info["previous_exemplar_article_char_start"] = None
        info["previous_exemplar_article_token_pos"] = None
        info["previous_exemplar_end_token_exclusive"] = None
        info["previous_exemplar_end_token_inclusive"] = None
        info["previous_exemplar_num_tokens_article_to_next_example"] = None
        if idx == 0:
            continue
        prev = results[idx - 1]
        content_char_start = int(prev.get("char_end", prev["char_start"]))
        search_end = int(info["char_start"])
        # Exclude the item header itself in positions="all", so numeric header tokens
        # do not trivially dominate the full-block average.
        while content_char_start < search_end and prompt[content_char_start] in " \t\r\n":
            content_char_start += 1
        content_tok = find_token_overlapping_char(offset_mapping, int(content_char_start))
        end_exclusive = int(info["example_token_pos"])
        if content_tok is None or int(content_tok) >= end_exclusive:
            continue
        info["previous_exemplar_marker_number"] = int(prev["marker_number"])
        info["previous_exemplar_article_char_start"] = int(content_char_start)
        info["previous_exemplar_article_token_pos"] = int(content_tok)
        info["previous_exemplar_end_token_exclusive"] = int(end_exclusive)
        info["previous_exemplar_end_token_inclusive"] = int(end_exclusive - 1)
        info["previous_exemplar_num_tokens_article_to_next_example"] = int(end_exclusive - int(content_tok))
    return results


def normalize_position_offsets(position_offsets):
    """
    Normalize user-selected token positions.

    Supported forms:
      positions = [-1]      -> hidden state at the token immediately before "Example".
      positions = [-2, -3]  -> encode both positions separately, then average SAE feature activations.
      positions = "all"    -> encode all tokens in the previous completed exemplar, from Article:
                               through the token immediately before the next Tool marker.
    """
    if is_all_position_selection(position_offsets):
        return ALL_EXEMPLAR_POSITION_MODE
    if isinstance(position_offsets, str):
        raise ValueError("positions must be 'all' or a non-empty iterable of negative integer offsets, e.g. [-1].")

    offsets = [int(p) for p in position_offsets]
    if len(offsets) == 0:
        raise ValueError("positions / POSITION_OFFSETS must contain at least one offset, e.g. [-1], or be 'all'.")
    if any(p >= 0 for p in offsets):
        raise ValueError(
            "All offsets in positions must be negative. Use -1 for the token right before Example, "
            "-2 for two tokens before Example, etc. Use positions='all' for full previous-exemplar spans."
        )
    if len(set(offsets)) != len(offsets):
        raise ValueError(f"positions contains duplicate offsets: {offsets}")
    return offsets


def build_measurement_position_rows(marker_info, input_ids_cpu, position_offsets):
    """
    Build per-marker token-position rows.

    For fixed offsets this returns rectangular rows such as [[pos-1], [pos-1], ...].
    For positions="all" this returns ragged rows: one row per marker, containing every token
    from the previous exemplar's Article: token through the token before the current Example token.
    """
    position_offsets = normalize_position_offsets(position_offsets)
    position_rows = []
    seq_len = len(input_ids_cpu)

    for info in marker_info:
        example_token_pos = int(info["example_token_pos"])

        if is_all_position_selection(position_offsets):
            start = info.get("previous_exemplar_article_token_pos", None)
            end_exclusive = info.get("previous_exemplar_end_token_exclusive", example_token_pos)
            if start is None or end_exclusive is None:
                raise ValueError(
                    "positions='all' requires a previous completed exemplar. "
                    "Use skip_first_marker=True, or use fixed negative offsets for Example 1."
                )
            start = int(start)
            end_exclusive = int(end_exclusive)
            positions_this_marker = list(range(start, end_exclusive))
            if len(positions_this_marker) == 0:
                raise ValueError(
                    f"positions='all' produced an empty span for marker={info['match']!r}: "
                    f"start={start}, end_exclusive={end_exclusive}"
                )

            info["position_offsets"] = ALL_EXEMPLAR_POSITION_MODE
            info["position_selection"] = "all_previous_exemplar_article_to_before_next_example"
            info["measurement_positions"] = f"{start}:{end_exclusive}"
            info["measurement_positions_format"] = "python_range_start_inclusive_end_exclusive"
            info["n_position_offsets"] = int(len(positions_this_marker))
            info["pred_pos"] = int(start)  # backward-compatible representative: first measured token.
            info["anchor_pred_pos"] = int(example_token_pos - 1)
            info["measurement_position_min"] = int(start)
            info["measurement_position_max"] = int(end_exclusive - 1)
        else:
            positions_this_marker = [example_token_pos + int(offset) for offset in position_offsets]
            info["anchor_pred_pos"] = int(example_token_pos - 1)
            info["pred_pos"] = int(positions_this_marker[0])
            info["position_offsets"] = ",".join(str(p) for p in position_offsets)
            info["position_selection"] = "fixed_negative_offsets_before_next_example"
            info["measurement_positions"] = ",".join(str(p) for p in positions_this_marker)
            info["measurement_positions_format"] = "comma_separated_token_indices"
            info["n_position_offsets"] = int(len(position_offsets))
            info["measurement_position_min"] = int(min(positions_this_marker))
            info["measurement_position_max"] = int(max(positions_this_marker))

        invalid_positions = [p for p in positions_this_marker if p < 0 or p >= seq_len]
        if len(invalid_positions) > 0:
            raise ValueError(
                "Requested positions go outside the tokenized prompt. "
                f"marker={info['match']!r}, example_token_pos={example_token_pos}, "
                f"position_offsets={position_offsets}, invalid_positions={invalid_positions}, "
                f"sequence_length={seq_len}"
            )
        position_rows.append(positions_this_marker)

    return marker_info, position_rows


def build_position_row_index(position_rows):
    """Convert a list[list[int]] of per-marker positions into flat indexing metadata."""
    if len(position_rows) == 0:
        raise ValueError("position_rows is empty")
    row_lengths = np.asarray([len(row) for row in position_rows], dtype=np.int64)
    if np.any(row_lengths <= 0):
        raise ValueError(f"Every marker must have at least one measured token. row_lengths={row_lengths.tolist()}")

    flat_positions = np.asarray([p for row in position_rows for p in row], dtype=np.int64)
    row_ids = np.repeat(np.arange(len(position_rows), dtype=np.int64), row_lengths)
    is_ragged = bool(len(set(row_lengths.tolist())) != 1)

    return {
        "position_rows": position_rows,
        "flat_positions": flat_positions,
        "row_ids": row_ids,
        "row_lengths": row_lengths,
        "n_markers": int(len(position_rows)),
        "n_offsets": None if is_ragged else int(row_lengths[0]),
        "is_ragged": is_ragged,
    }


def flatten_positions_for_rows(position_index, rows=None):
    """Return flat token positions for all rows or a selected subset of marker rows."""
    flat_positions = np.asarray(position_index["flat_positions"], dtype=np.int64)
    row_ids = np.asarray(position_index["row_ids"], dtype=np.int64)
    if rows is None:
        return flat_positions
    rows = np.asarray(list(rows), dtype=np.int64)
    if rows.size == 0:
        return np.asarray([], dtype=np.int64)
    return flat_positions[np.isin(row_ids, rows)]


def make_selected_hidden_from_flat(selected_flat: torch.Tensor, position_index):
    """
    Convert flat selected states [sum(row_lengths), d_model] into either:
      - a rectangular tensor [n_markers, n_offsets, d_model], or
      - a ragged-selection dict for variable-length positions='all' spans.
    """
    if bool(position_index.get("is_ragged", False)):
        return {
            "flat_hidden": selected_flat,
            "row_ids": torch.as_tensor(position_index["row_ids"], dtype=torch.long),
            "row_lengths": torch.as_tensor(position_index["row_lengths"], dtype=torch.float32),
            "n_markers": int(position_index["n_markers"]),
        }
    n_markers = int(position_index["n_markers"])
    n_offsets = int(position_index["n_offsets"])
    return selected_flat.reshape(n_markers, n_offsets, selected_flat.shape[-1])


def is_ragged_selected_hidden(selected_hidden) -> bool:
    return isinstance(selected_hidden, dict) and "flat_hidden" in selected_hidden


def selected_hidden_tensor(selected_hidden):
    return selected_hidden["flat_hidden"] if is_ragged_selected_hidden(selected_hidden) else selected_hidden


def mean_values_over_selected_positions(values: torch.Tensor, selected_hidden):
    """
    Average per-token values over the selected positions for each marker.

    Rectangular case:
      values shape [n_markers, n_offsets, ...] -> mean over dim=1.
    Ragged case:
      values shape [sum(row_lengths), ...] -> index-add mean using row_ids.
    """
    if is_ragged_selected_hidden(selected_hidden):
        n_markers = int(selected_hidden["n_markers"])
        row_ids = selected_hidden["row_ids"].to(device=values.device, dtype=torch.long)
        row_lengths = selected_hidden["row_lengths"].to(device=values.device, dtype=values.dtype).clamp_min(1)
        out_shape = (n_markers,) + tuple(values.shape[1:])
        out = torch.zeros(out_shape, device=values.device, dtype=values.dtype)
        out.index_add_(0, row_ids, values)
        denom = row_lengths
        while denom.ndim < out.ndim:
            denom = denom.unsqueeze(-1)
        return out / denom

    if values.ndim >= 2:
        return values.mean(dim=1)
    return values


@torch.inference_mode()
def capture_hidden_at_example_markers_multi_layer(
    prompt,
    target_layers=TARGET_LAYERS,
    skip_first_marker=True,
    position_offsets=POSITION_OFFSETS,
):
    """
    Run the model once on one prompt and capture residual states at all requested layers.

    `position_offsets` may be either:
      - a list of negative offsets relative to the first token of the next "Example" marker; or
      - "all", which averages over every token in the previous completed exemplar from Article:
        through the token immediately before the next Tool marker.

    Returns:
      marker_info: list of metadata for measured Tool markers
      selected_hidden_by_layer: dict[layer] -> rectangular tensor or ragged-selection dict

    This is the efficient layer-sweep path: one forward pass per prompt, not one forward pass per layer.
    """
    target_layers = [int(layer) for layer in target_layers]
    position_offsets = normalize_position_offsets(position_offsets)
    input_device = get_input_device(model)

    # Prompt is already rendered and explicitly BOS-prefixed when ADD_BOS_TOKEN=True.
    # Keep add_special_tokens=False to avoid duplicate special tokens and preserve rendered-string offsets.
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=False,
    )

    offset_mapping = enc.pop("offset_mapping")[0].tolist()
    input_ids_cpu = enc["input_ids"][0].tolist()

    marker_info = get_example_marker_prediction_positions(
        prompt=prompt,
        input_ids_cpu=input_ids_cpu,
        offset_mapping=offset_mapping,
    )

    if skip_first_marker:
        marker_info = marker_info[1:]

    if len(marker_info) == 0:
        return marker_info, {}

    marker_info, position_rows = build_measurement_position_rows(
        marker_info=marker_info,
        input_ids_cpu=input_ids_cpu,
        position_offsets=position_offsets,
    )
    position_index = build_position_row_index(position_rows)
    flat_positions_cpu = position_index["flat_positions"].astype(int).tolist()

    input_ids = enc["input_ids"].to(input_device)
    attention_mask = enc["attention_mask"].to(input_device)

    selected_hidden_by_layer = {}
    handles = []

    def make_hook(layer: int):
        def target_layer_hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            idx = torch.tensor(flat_positions_cpu, device=x.device, dtype=torch.long)
            selected_flat = x[0, idx, :]
            selected_hidden_by_layer[int(layer)] = make_selected_hidden_from_flat(
                selected_flat.detach().contiguous(),
                position_index,
            )
            return output
        return target_layer_hook

    try:
        for layer in target_layers:
            target_block = model.get_submodule(f"model.layers.{int(layer)}")
            handles.append(target_block.register_forward_hook(make_hook(int(layer))))

        _ = get_backbone_model(model)(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        for handle in handles:
            handle.remove()

    missing_layers = sorted(set(target_layers) - set(selected_hidden_by_layer.keys()))
    if len(missing_layers) > 0:
        raise RuntimeError(f"Failed to capture hidden states for layers: {missing_layers}")

    return marker_info, selected_hidden_by_layer


@torch.inference_mode()
def _call_sae_encode_post_activation(sae, x_flat: torch.Tensor) -> torch.Tensor:
    """Call an SAE's standard encoder and normalize output shape to [tokens, features]."""
    try:
        feats = sae.encode(x_flat.unsqueeze(0))
        if isinstance(feats, tuple):
            feats = feats[0]
        if feats.ndim == 3:
            feats = feats.squeeze(0)
    except Exception:
        feats = sae.encode(x_flat)
        if isinstance(feats, tuple):
            feats = feats[0]
    if feats.ndim != 2:
        raise ValueError(f"Unexpected SAE feature activation shape: {tuple(feats.shape)}")
    return feats


def _gemma_scope_encode_pre(sae, x_flat: torch.Tensor) -> torch.Tensor:
    """
    Best-effort SAE Lens pre-activation extraction for Gemma Scope SAEs.
    This mirrors the usual SAE Lens encode path up to, but not including, the activation function.
    """
    if hasattr(sae, "encode_pre"):
        pre = sae.encode_pre(x_flat)
        if isinstance(pre, tuple):
            pre = pre[0]
        return pre.squeeze(0) if getattr(pre, "ndim", 0) == 3 else pre

    if hasattr(sae, "process_sae_in"):
        sae_in = sae.process_sae_in(x_flat)
    elif hasattr(sae, "b_dec"):
        sae_in = x_flat - sae.b_dec
    else:
        sae_in = x_flat

    if not hasattr(sae, "W_enc") or not hasattr(sae, "b_enc"):
        raise AttributeError(
            "Could not extract Gemma Scope pre-activations: SAE object has neither encode_pre nor W_enc/b_enc. "
            "Set GEMMA_SCOPE_FEATURE_VALUE_MODE='activation' to use the standard SAE Lens encoder."
        )

    W_enc = sae.W_enc
    b_enc = sae.b_enc
    if W_enc.shape[0] == sae_in.shape[-1]:
        pre = sae_in @ W_enc + b_enc
    elif W_enc.shape[1] == sae_in.shape[-1]:
        pre = sae_in @ W_enc.T + b_enc
    else:
        raise ValueError(f"Unexpected Gemma Scope W_enc shape {tuple(W_enc.shape)} for input dim {sae_in.shape[-1]}")

    # Preserve SAE Lens hooks when available, but do not require them.
    if hasattr(sae, "hook_sae_acts_pre"):
        try:
            pre = sae.hook_sae_acts_pre(pre)
        except Exception:
            pass
    return pre


def _llama_scope_encode_pre(sae, x_flat: torch.Tensor) -> torch.Tensor:
    """Pre-activation extraction for the direct Llama Scope wrapper."""
    x = x_flat.to(device=sae.device, dtype=sae.dtype)
    hidden_pre = x @ sae.W_E + sae.b_E
    if getattr(sae, "sparsity_include_decoder_norm", False):
        decoder_norm = torch.norm(sae.W_D, p=2, dim=1).clamp_min(1e-8)
        hidden_pre = hidden_pre * decoder_norm
    return hidden_pre


def _qwen_encode_pre_and_activation(sae, x_flat: torch.Tensor):
    pre = sae.encode_pre(x_flat)
    activation = sae.activation_fn(pre)
    return pre, activation


def _encode_sae_feature_tensors_flat(sae, x_flat: torch.Tensor):
    """Return (configured_feature_values, post_threshold_values) for a flat [tokens, d_model] tensor."""
    if SAE_PROVIDER == "qwen_batchtopk_direct":
        pre, post = _qwen_encode_pre_and_activation(sae, x_flat)
        mode = QWEN_FEATURE_VALUE_MODE
        if mode == "preactivation":
            feats = pre
        elif mode == "activation":
            feats = post
        else:
            raise ValueError(f"Unknown QWEN_FEATURE_VALUE_MODE={mode!r}")
        active_feats = post

    elif SAE_PROVIDER in {"saelens", "gemma_scope_saelens"}:
        post = _call_sae_encode_post_activation(sae, x_flat)
        mode = SAELENS_FEATURE_VALUE_MODE
        if mode == "preactivation":
            feats = _gemma_scope_encode_pre(sae, x_flat)
        elif mode == "activation":
            feats = post
        else:
            raise ValueError(f"Unknown SAELENS_FEATURE_VALUE_MODE={mode!r}")
        active_feats = post

    elif SAE_PROVIDER == "llama_scope_direct":
        post = _call_sae_encode_post_activation(sae, x_flat)
        mode = LLAMA_SCOPE_FEATURE_VALUE_MODE
        if mode == "preactivation":
            feats = _llama_scope_encode_pre(sae, x_flat)
        elif mode == "activation":
            feats = post
        else:
            raise ValueError(f"Unknown LLAMA_SCOPE_FEATURE_VALUE_MODE={mode!r}")
        active_feats = post

    else:
        feats = _call_sae_encode_post_activation(sae, x_flat)
        active_feats = feats

    if feats.ndim != 2:
        raise ValueError(f"Unexpected SAE feature value shape: {tuple(feats.shape)}")
    if active_feats.ndim != 2:
        raise ValueError(f"Unexpected SAE active-feature shape: {tuple(active_feats.shape)}")
    return feats, active_feats


def encode_sae_features(sae, selected_hidden: torch.Tensor):
    """
    Encode residual-stream activations through the configured SAE.

    Input forms:
      - [num_markers, d_model] for a single token position;
      - [num_markers, n_offsets, d_model] for fixed multiple token offsets;
      - ragged-selection dict for positions="all", with flat_hidden and row_ids.

    Returns:
      feature_values: [num_markers, n_features]
        SAE feature values averaged over requested token positions.
      active_values: [num_markers, n_features]
        Mean post-threshold active indicator over requested token positions.
    """
    if selected_hidden is None:
        return None, None

    if is_ragged_selected_hidden(selected_hidden):
        flat_hidden = selected_hidden["flat_hidden"]
        sae_device = get_sae_device(sae, fallback=flat_hidden.device)
        sae_dtype = get_sae_dtype(sae, fallback=SAE_DTYPE)
        x_flat = flat_hidden.to(device=sae_device, dtype=sae_dtype)

        n_markers = int(selected_hidden["n_markers"])
        row_ids_all = selected_hidden["row_ids"].to(device=sae_device, dtype=torch.long)
        row_lengths = selected_hidden["row_lengths"].to(device=sae_device, dtype=torch.float32).clamp_min(1)

        chunk_size = globals().get("SAE_ENCODE_CHUNK_SIZE", None)
        if chunk_size is None or int(chunk_size) <= 0:
            chunk_size = int(x_flat.shape[0])
        chunk_size = int(chunk_size)

        feature_sums = None
        active_sums = None
        for start in range(0, int(x_flat.shape[0]), chunk_size):
            end = min(start + chunk_size, int(x_flat.shape[0]))
            feats, active_feats = _encode_sae_feature_tensors_flat(sae, x_flat[start:end])
            feats = feats.float()
            active_indicator = (active_feats > 0).to(dtype=torch.float32)
            rows = row_ids_all[start:end]

            if feature_sums is None:
                n_features = int(feats.shape[-1])
                feature_sums = torch.zeros((n_markers, n_features), device=feats.device, dtype=torch.float32)
                active_sums = torch.zeros((n_markers, n_features), device=feats.device, dtype=torch.float32)

            feature_sums.index_add_(0, rows, feats)
            active_sums.index_add_(0, rows, active_indicator)

            del feats, active_feats, active_indicator, rows

        denom = row_lengths.unsqueeze(-1)
        feature_values = feature_sums / denom
        active_values = active_sums / denom
        return feature_values, active_values

    sae_device = get_sae_device(sae, fallback=selected_hidden.device)
    sae_dtype = get_sae_dtype(sae, fallback=SAE_DTYPE)
    x = selected_hidden.to(device=sae_device, dtype=sae_dtype)

    if x.ndim == 2:
        n_markers, d_model = x.shape
        n_offsets = 1
        x_flat = x
    elif x.ndim == 3:
        n_markers, n_offsets, d_model = x.shape
        x_flat = x.reshape(n_markers * n_offsets, d_model)
    else:
        raise ValueError(f"Expected selected_hidden to be 2D, 3D, or ragged dict, got shape {tuple(x.shape)}")

    feats, active_feats = _encode_sae_feature_tensors_flat(sae, x_flat)

    if n_offsets == 1 and x.ndim == 2:
        feature_values = feats
        active_values = (active_feats > 0).to(dtype=torch.float32)
    else:
        feats_by_offset = feats.reshape(n_markers, n_offsets, -1)
        active_by_offset = active_feats.reshape(n_markers, n_offsets, -1)
        feature_values = feats_by_offset.mean(dim=1)
        active_values = (active_by_offset > 0).to(dtype=torch.float32).mean(dim=1)

    return feature_values, active_values

@torch.inference_mode()
def compute_sae_features_at_example_markers_multi_layer(
    prompt,
    saes_by_layer: Dict[int, Any],
    skip_first_marker=True,
    position_offsets=POSITION_OFFSETS,
):
    """
    Run one model forward for `prompt`, then encode the captured residuals through each layer's SAE.

    Returns:
      marker_info
      feature_cpu_by_layer: dict[layer] -> [num_markers, n_features] float32 CPU tensor
      active_cpu_by_layer: dict[layer] -> [num_markers, n_features] float32 CPU tensor

    With multiple offsets, feature activations are averaged over offsets after SAE encoding.
    The active tensor is the mean active indicator over offsets.
    """
    layers = sorted(int(layer) for layer in saes_by_layer.keys())
    marker_info, selected_hidden_by_layer = capture_hidden_at_example_markers_multi_layer(
        prompt=prompt,
        target_layers=layers,
        skip_first_marker=skip_first_marker,
        position_offsets=position_offsets,
    )

    if len(marker_info) == 0:
        return marker_info, {}, {}

    feature_cpu_by_layer = {}
    active_cpu_by_layer = {}

    for layer in layers:
        selected_hidden = selected_hidden_by_layer.get(layer)
        if selected_hidden is None:
            raise RuntimeError(f"Missing captured hidden state for layer {layer}")

        feature_values, active_values = encode_sae_features(saes_by_layer[layer], selected_hidden)

        if feature_values.shape[0] != len(marker_info):
            raise RuntimeError(
                f"Layer {layer}: SAE feature row count {feature_values.shape[0]} does not match "
                f"marker_info length {len(marker_info)}."
            )

        feature_cpu_by_layer[layer] = feature_values.detach().float().cpu()
        active_cpu_by_layer[layer] = active_values.detach().float().cpu()

        del selected_hidden, feature_values, active_values

    del selected_hidden_by_layer
    return marker_info, feature_cpu_by_layer, active_cpu_by_layer


In [ ]:
# ============================================================
# 3. Run layer sweep efficiently: one model forward per prompt, all requested layers captured
# ============================================================


def _new_accumulator(n_markers: int, n_features: int):
    return {
        "sum_by_marker": np.zeros((n_markers, n_features), dtype=np.float64),
        "sumsq_by_marker": np.zeros((n_markers, n_features), dtype=np.float64),
        "active_by_marker": np.zeros((n_markers, n_features), dtype=np.float64),
        "count_by_marker": np.zeros(n_markers, dtype=np.int64),
    }


def _update_accumulator(acc, mrow: int, vec: np.ndarray, active_vec: np.ndarray):
    acc["sum_by_marker"][mrow] += vec
    acc["sumsq_by_marker"][mrow] += vec * vec
    acc["active_by_marker"][mrow] += active_vec
    acc["count_by_marker"][mrow] += 1


def save_layer_outputs(layer: int, layer_output: Dict[str, Any]):
    layer_dir = os.path.join(OUTPUT_DIR, f"layer_{int(layer)}")
    os.makedirs(layer_dir, exist_ok=True)

    np.save(os.path.join(layer_dir, "marker_numbers.npy"), layer_output["marker_numbers"])
    for split_name, acc in layer_output["accumulators"].items():
        for key, value in acc.items():
            np.save(os.path.join(layer_dir, f"{split_name}_{key}.npy"), value)

    layer_output["marker_records_df"].to_csv(os.path.join(layer_dir, "marker_records_df.csv"), index=False)

    config = {
        "model_name": MODEL_NAME,
        "model_label": MODEL_LABEL,
        "add_bos_token": ADD_BOS_TOKEN,
        "use_chat_template": USE_CHAT_TEMPLATE,
        "chat_template_kwargs": CHAT_TEMPLATE_KWARGS,
        "chat_template_use_system_role": CHAT_TEMPLATE_USE_SYSTEM_ROLE,
        "agnews_system_prompt": AGNEWS_SYSTEM_PROMPT,
        "prompt_format": PROMPT_FORMAT_NAME,
        "target_layer": int(layer),
        "sweep_mode": "single_forward_multi_layer_hooks",
        "target_layers_swept_together": [int(x) for x in TARGET_LAYERS],
        "sae_provider": SAE_PROVIDER,
        "sae_description": describe_sae_for_layer(int(layer)),
        "n_shots": N_SHOTS,
        "expected_num_examples": EXPECTED_NUM_EXAMPLES,
        "num_prompts_requested": NUM_PROMPTS,
        "max_prompts": MAX_PROMPTS,
        "target_marker_numbers": TARGET_MARKER_NUMBERS,
        "modulo_target_completed_counts": MODULO_TARGET_COMPLETED_COUNTS,
        "position_offsets": position_selection_to_jsonable(layer_output.get("position_offsets", POSITION_OFFSETS)),
        "position_selection_label": position_selection_label(layer_output.get("position_offsets", POSITION_OFFSETS)),
        "position_aggregation": "mean_of_sae_feature_activations_after_encoding",
        "sae_value_name": SAE_VALUE_NAME,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(os.path.join(layer_dir, "sweep_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    return layer_dir


@torch.inference_mode()
def run_sae_example_marker_position_sweep_all_layers_single_forward(
    prompts,
    *,
    layers=TARGET_LAYERS,
    target_marker_numbers=TARGET_MARKER_NUMBERS,
    skip_first_marker=True,
    max_prompts=MAX_PROMPTS,
    verbose_every=VERBOSE_EVERY,
    empty_cache_every=EMPTY_CACHE_EVERY,
    position_offsets=POSITION_OFFSETS,
):
    """
    Accumulate SAE feature activations at all "Before Tool N" positions for all requested layers.

    Efficiency: this function runs each prompt through the model exactly once. Forward hooks capture
    the selected residual states at every layer in `layers`, then each captured tensor is passed
    through its corresponding SAE.

    Deterministic split accumulators are also kept:
      - select: even prompt indices
      - eval: odd prompt indices
    These enable a cross-validated layer-level modulo-10 metric.
    """
    layers = [int(layer) for layer in layers]
    position_offsets = normalize_position_offsets(position_offsets)
    prompts_to_run = prompts[:max_prompts] if max_prompts is not None else prompts
    expected_marker_numbers = list(range(2 if skip_first_marker else 1, EXPECTED_NUM_EXAMPLES + 1))
    marker_to_row = {m: i for i, m in enumerate(expected_marker_numbers)}
    n_markers = len(expected_marker_numbers)
    target_marker_set = set(int(x) for x in target_marker_numbers)
    modulo_target_completed_count_set = set(int(x) for x in MODULO_TARGET_COMPLETED_COUNTS)

    print("Loading all requested SAEs once so that each prompt uses one model forward pass.")
    print("position selection:", position_selection_label(position_offsets))
    saes_by_layer = load_all_saes_for_layers(layers)

    accumulators_by_layer = {layer: None for layer in layers}
    records_by_layer = {layer: [] for layer in layers}
    t0 = time.time()

    iterator = enumerate(prompts_to_run)
    if tqdm is not None:
        iterator = tqdm(iterator, total=len(prompts_to_run), desc="Single-forward multi-layer SAE Tool-marker sweep")

    for prompt_idx, prompt in iterator:
        marker_info, feature_cpu_by_layer, active_cpu_by_layer = compute_sae_features_at_example_markers_multi_layer(
            prompt,
            saes_by_layer=saes_by_layer,
            skip_first_marker=skip_first_marker,
            position_offsets=position_offsets,
        )

        if len(marker_info) == 0:
            continue

        split_name = "select" if (int(prompt_idx) % 2 == 0) else "eval"

        # Same marker metadata for every layer; feature activations differ by layer.
        for layer in layers:
            feat_cpu = feature_cpu_by_layer.get(layer)
            active_cpu = active_cpu_by_layer.get(layer)
            if feat_cpu is None or active_cpu is None:
                raise RuntimeError(f"Missing SAE features for layer {layer}")

            feat_np = feat_cpu.numpy().astype(np.float64, copy=False)
            active_np = active_cpu.numpy().astype(np.float64, copy=False)

            if accumulators_by_layer[layer] is None:
                n_features = int(feat_np.shape[-1])
                accumulators_by_layer[layer] = {
                    "all": _new_accumulator(n_markers, n_features),
                    "select": _new_accumulator(n_markers, n_features),
                    "eval": _new_accumulator(n_markers, n_features),
                }
                print(f"Layer {layer}: initialized accumulators with n_features={n_features}")
                if layer == layers[0]:
                    print("First few marker infos:")
                    for info in marker_info[:5]:
                        print(info)

            for row_idx, info in enumerate(marker_info):
                marker_number = int(info["marker_number"])
                if marker_number not in marker_to_row:
                    continue
                mrow = marker_to_row[marker_number]
                vec = feat_np[row_idx]
                avec = active_np[row_idx]

                _update_accumulator(accumulators_by_layer[layer]["all"], mrow, vec, avec)
                _update_accumulator(accumulators_by_layer[layer][split_name], mrow, vec, avec)

                records_by_layer[layer].append({
                    "layer": int(layer),
                    "prompt_idx": int(prompt_idx),
                    "split": split_name,
                    "marker_number": marker_number,
                    "completed_examples": int(info["completed_examples"]),
                    "pred_pos": int(info["pred_pos"]),
                    "example_token_pos": int(info["example_token_pos"]),
                    "anchor_pred_pos": int(info["anchor_pred_pos"]),
                    "position_offsets": info["position_offsets"],
                    "measurement_positions": info["measurement_positions"],
                    "n_position_offsets": int(info["n_position_offsets"]),
                    "position_selection": info.get("position_selection", ""),
                    "measurement_positions_format": info.get("measurement_positions_format", ""),
                    "previous_exemplar_marker_number": info.get("previous_exemplar_marker_number", None),
                    "previous_exemplar_num_tokens_article_to_next_example": info.get("previous_exemplar_num_tokens_article_to_next_example", None),
                    "measurement_position_min": int(info["measurement_position_min"]),
                    "measurement_position_max": int(info["measurement_position_max"]),
                    "target_id": int(info["target_id"]),
                    "target_decoded": info["target_decoded"],
                    "match": info["match"],
                    "is_target_marker": marker_number in target_marker_set,
                    "is_modulo10_completed_count": int(info["completed_examples"]) in modulo_target_completed_count_set,
                })

            del feat_cpu, active_cpu, feat_np, active_np

        del feature_cpu_by_layer, active_cpu_by_layer

        if verbose_every is not None and (prompt_idx + 1) % verbose_every == 0:
            elapsed = time.time() - t0
            print(
                f"processed {prompt_idx + 1}/{len(prompts_to_run)} prompts "
                f"with layers={layers} | elapsed={elapsed:.1f}s | avg={elapsed / (prompt_idx + 1):.3f}s/prompt"
            )

        if torch.cuda.is_available() and empty_cache_every is not None and (prompt_idx + 1) % empty_cache_every == 0:
            gc.collect()
            torch.cuda.empty_cache()

    layer_sweep_outputs = {}
    for layer in layers:
        if accumulators_by_layer[layer] is None:
            raise RuntimeError(f"Layer {layer}: no SAE activations were collected.")

        marker_records_df = pd.DataFrame(records_by_layer[layer])
        layer_output = {
            "layer": int(layer),
            "marker_numbers": np.asarray(expected_marker_numbers, dtype=np.int64),
            "accumulators": accumulators_by_layer[layer],
            "marker_records_df": marker_records_df,
            "position_offsets": position_selection_to_jsonable(position_offsets),
        }
        layer_sweep_outputs[layer] = layer_output

        layer_dir = save_layer_outputs(layer, layer_output)
        print("=" * 100)
        print(f"Layer {layer} SAE Tool-marker position sweep complete")
        print("sweep mode: single_forward_multi_layer_hooks")
        print("num prompts:", len(prompts_to_run))
        print("num marker records:", len(marker_records_df))
        print("marker numbers:", expected_marker_numbers[:10], "...", expected_marker_numbers[-10:])
        print("target marker numbers:", target_marker_numbers)
        print("position selection:", position_selection_label(position_offsets))
        print("saved to:", layer_dir)
        print("=" * 100)

    print("Finished single-forward sweep for layers:", sorted(layer_sweep_outputs.keys()))
    print("elapsed seconds:", f"{time.time() - t0:.2f}")

    # Release all SAEs after the sweep.
    del saes_by_layer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return layer_sweep_outputs


RUN_LAYER_SWEEP = True

layer_sweep_outputs = {}
if RUN_LAYER_SWEEP:
    layer_sweep_outputs = run_sae_example_marker_position_sweep_all_layers_single_forward(
        prompts=original_prompts,
        layers=TARGET_LAYERS,
        target_marker_numbers=TARGET_MARKER_NUMBERS,
        skip_first_marker=True,
        max_prompts=MAX_PROMPTS,
        verbose_every=VERBOSE_EVERY,
        empty_cache_every=EMPTY_CACHE_EVERY,
        position_offsets=POSITION_OFFSETS,
    )
else:
    print("RUN_LAYER_SWEEP=False. Use the next cell to load cached arrays.")


In [ ]:

# ============================================================
# 4. Optional: load cached layer-sweep arrays instead of recomputing
# ============================================================


def load_layer_outputs_from_cache(layers=TARGET_LAYERS):
    loaded = {}
    for layer in layers:
        layer = int(layer)
        layer_dir = os.path.join(OUTPUT_DIR, f"layer_{layer}")
        if not os.path.isdir(layer_dir):
            raise FileNotFoundError(f"Missing cache directory for layer {layer}: {layer_dir}")

        marker_numbers = np.load(os.path.join(layer_dir, "marker_numbers.npy"))
        accumulators = {}
        for split_name in ["all", "select", "eval"]:
            accumulators[split_name] = {
                "sum_by_marker": np.load(os.path.join(layer_dir, f"{split_name}_sum_by_marker.npy")),
                "sumsq_by_marker": np.load(os.path.join(layer_dir, f"{split_name}_sumsq_by_marker.npy")),
                "active_by_marker": np.load(os.path.join(layer_dir, f"{split_name}_active_by_marker.npy")),
                "count_by_marker": np.load(os.path.join(layer_dir, f"{split_name}_count_by_marker.npy")),
            }
        marker_records_df = pd.read_csv(os.path.join(layer_dir, "marker_records_df.csv"))
        loaded[layer] = {
            "layer": layer,
            "marker_numbers": marker_numbers,
            "accumulators": accumulators,
            "marker_records_df": marker_records_df,
        }
        print(f"Loaded layer {layer} cached arrays from {layer_dir}")
    return loaded


# Uncomment if needed:
# layer_sweep_outputs = load_layer_outputs_from_cache(TARGET_LAYERS)


In [ ]:

# ============================================================
# 5. Rank features: modulo-10 target positions vs other Tool-marker positions, per layer
# ============================================================

EPS = 1e-12


def compute_marker_stats(layer_output: Dict[str, Any], split_name: str = "all"):
    acc = layer_output["accumulators"][split_name]
    marker_numbers = np.asarray(layer_output["marker_numbers"], dtype=np.int64)
    counts = np.asarray(acc["count_by_marker"], dtype=np.float64)
    sum_by_marker = acc["sum_by_marker"]
    sumsq_by_marker = acc["sumsq_by_marker"]
    active_by_marker = acc["active_by_marker"]

    if np.any(counts <= 0):
        missing = marker_numbers[counts <= 0].tolist()
        print(f"Warning: split={split_name} has marker numbers with zero count:", missing)

    mean_by_marker = sum_by_marker / np.maximum(counts[:, None], 1.0)
    var_by_marker = (sumsq_by_marker - (sum_by_marker ** 2) / np.maximum(counts[:, None], 1.0)) / np.maximum(counts[:, None] - 1.0, 1.0)
    var_by_marker = np.maximum(var_by_marker, 0.0)
    active_freq_by_marker = active_by_marker / np.maximum(counts[:, None], 1.0)

    return {
        "marker_numbers": marker_numbers,
        "counts": counts,
        "sum_by_marker": sum_by_marker,
        "sumsq_by_marker": sumsq_by_marker,
        "active_by_marker": active_by_marker,
        "mean_by_marker": mean_by_marker,
        "var_by_marker": var_by_marker,
        "active_freq_by_marker": active_freq_by_marker,
    }


def _aggregate_marker_group(stats: Dict[str, Any], mask):
    counts = stats["counts"]
    sum_by_marker = stats["sum_by_marker"]
    sumsq_by_marker = stats["sumsq_by_marker"]
    active_by_marker = stats["active_by_marker"]

    group_counts = counts[mask]
    group_sum = sum_by_marker[mask].sum(axis=0)
    group_sumsq = sumsq_by_marker[mask].sum(axis=0)
    group_active = active_by_marker[mask].sum(axis=0)
    n = float(group_counts.sum())
    mean = group_sum / max(n, 1.0)
    var = (group_sumsq - (group_sum ** 2) / max(n, 1.0)) / max(n - 1.0, 1.0)
    var = np.maximum(var, 0.0)
    active_freq = group_active / max(n, 1.0)
    return n, mean, var, active_freq


def welch_t_and_d(mean_a, var_a, n_a, mean_b, var_b, n_b):
    diff = mean_a - mean_b
    se = np.sqrt(var_a / max(n_a, 1.0) + var_b / max(n_b, 1.0) + EPS)
    t = diff / se
    pooled_sd = np.sqrt(((n_a - 1.0) * var_a + (n_b - 1.0) * var_b) / max(n_a + n_b - 2.0, 1.0) + EPS)
    d = diff / pooled_sd
    return diff, t, d


def build_group_contrast_df(stats: Dict[str, Any], target_mask, other_mask, contrast_name="target_vs_other"):
    mean_by_marker = stats["mean_by_marker"]

    n_t, mean_t, var_t, active_t = _aggregate_marker_group(stats, target_mask)
    n_o, mean_o, var_o, active_o = _aggregate_marker_group(stats, other_mask)
    diff, welch_t, cohen_d = welch_t_and_d(mean_t, var_t, n_t, mean_o, var_o, n_o)

    target_marker_mean_max = mean_by_marker[target_mask].max(axis=0)
    target_marker_mean_min = mean_by_marker[target_mask].min(axis=0)
    other_marker_mean_max = mean_by_marker[other_mask].max(axis=0)
    other_marker_mean_min = mean_by_marker[other_mask].min(axis=0)

    target_peak_advantage = target_marker_mean_max - other_marker_mean_max
    target_trough_advantage = other_marker_mean_min - target_marker_mean_min

    df = pd.DataFrame({
        "feature_idx": np.arange(mean_t.shape[0], dtype=int),
        "contrast_name": contrast_name,
        "n_target_observations": int(n_t),
        "n_other_observations": int(n_o),
        "target_mean_activation": mean_t,
        "other_mean_activation": mean_o,
        "target_minus_other_mean": diff,
        "welch_t_target_gt_other": welch_t,
        "abs_welch_t": np.abs(welch_t),
        "cohen_d_target_vs_other": cohen_d,
        "target_active_freq": active_t,
        "other_active_freq": active_o,
        "target_minus_other_active_freq": active_t - active_o,
        "target_marker_mean_max": target_marker_mean_max,
        "target_marker_mean_min": target_marker_mean_min,
        "other_marker_mean_max": other_marker_mean_max,
        "other_marker_mean_min": other_marker_mean_min,
        "target_peak_advantage": target_peak_advantage,
        "target_trough_advantage": target_trough_advantage,
    })

    df["stronger_score"] = df["target_minus_other_mean"] * np.maximum(df["welch_t_target_gt_other"], 0.0)
    df["weaker_score"] = (-df["target_minus_other_mean"]) * np.maximum(-df["welch_t_target_gt_other"], 0.0)
    return df


def analyze_layer(layer: int, layer_output: Dict[str, Any]):
    layer = int(layer)
    stats = compute_marker_stats(layer_output, split_name="all")
    marker_numbers = stats["marker_numbers"]
    completed_counts = marker_numbers - 1

    target_marker_set = set(int(x) for x in TARGET_MARKER_NUMBERS)
    target_marker_mask = np.asarray([int(m) in target_marker_set for m in marker_numbers], dtype=bool)
    other_marker_mask = ~target_marker_mask

    print("=" * 110)
    print(f"Layer {layer}: target markers present:", marker_numbers[target_marker_mask].tolist())
    print(f"Layer {layer}: target completed counts:", completed_counts[target_marker_mask].tolist())
    print(f"Layer {layer}: other markers count:", int(other_marker_mask.sum()))

    combined_contrast_df = build_group_contrast_df(
        stats,
        target_marker_mask,
        other_marker_mask,
        contrast_name=f"layer_{layer}_modulo10_completed_count_markers_vs_other_markers",
    )

    combined_stronger_rank_df = combined_contrast_df.sort_values(
        ["welch_t_target_gt_other", "target_minus_other_mean", "cohen_d_target_vs_other"],
        ascending=False,
    ).reset_index(drop=True)

    combined_weaker_rank_df = combined_contrast_df.sort_values(
        ["welch_t_target_gt_other", "target_minus_other_mean", "cohen_d_target_vs_other"],
        ascending=True,
    ).reset_index(drop=True)

    layer_dir = os.path.join(OUTPUT_DIR, f"layer_{layer}")
    combined_contrast_df.to_csv(os.path.join(layer_dir, "combined_modulo10_marker_vs_other_feature_contrast.csv"), index=False)
    combined_stronger_rank_df.to_csv(os.path.join(layer_dir, "combined_modulo10_marker_stronger_rank_df.csv"), index=False)
    combined_weaker_rank_df.to_csv(os.path.join(layer_dir, "combined_modulo10_marker_weaker_rank_df.csv"), index=False)

    print(f"Layer {layer}: top features STRONGER at modulo-10 completed-count positions")
    display(combined_stronger_rank_df.head(50))
    print(f"Layer {layer}: top features WEAKER at modulo-10 completed-count positions")
    display(combined_weaker_rank_df.head(50))

    return {
        "layer": layer,
        "stats": stats,
        "target_marker_mask": target_marker_mask,
        "other_marker_mask": other_marker_mask,
        "combined_contrast_df": combined_contrast_df,
        "combined_stronger_rank_df": combined_stronger_rank_df,
        "combined_weaker_rank_df": combined_weaker_rank_df,
    }


analysis_by_layer = {}
for layer in TARGET_LAYERS:
    analysis_by_layer[int(layer)] = analyze_layer(int(layer), layer_sweep_outputs[int(layer)])

print("Saved per-layer contrast CSVs to:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 5b. Rank features by target-minus-other active frequency, per layer
# ============================================================

# This cell creates one dataframe per tested layer, sorted by target_minus_other_active_freq.
# Positive values mean the feature is active more often at modulo-10 completed-count positions
# than at all other measured marker positions.
# With multiple `positions`, active frequency is the mean per-offset active indicator.

active_freq_rank_dfs_by_layer = {}

for layer in TARGET_LAYERS:
    layer = int(layer)
    layer_dir = os.path.join(OUTPUT_DIR, f"layer_{layer}")
    df = analysis_by_layer[layer]["combined_contrast_df"].copy()

    active_freq_rank_df = df.sort_values(
        [
            "target_minus_other_active_freq",
            "target_active_freq",
            "welch_t_target_gt_other",
            "target_minus_other_mean",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    active_freq_rank_df.insert(0, "active_freq_rank", np.arange(1, len(active_freq_rank_df) + 1, dtype=int))
    active_freq_rank_dfs_by_layer[layer] = active_freq_rank_df

    # Convenience variables, e.g. active_freq_rank_df_layer_10, active_freq_rank_df_layer_20, ...
    globals()[f"active_freq_rank_df_layer_{layer}"] = active_freq_rank_df

    out_path = os.path.join(layer_dir, "combined_modulo10_marker_active_freq_rank_df.csv")
    active_freq_rank_df.to_csv(out_path, index=False)

    print("=" * 110)
    print(f"Layer {layer}: features sorted by target_minus_other_active_freq")
    print("Saved:", out_path)
    display(active_freq_rank_df.head(50))

print("Available dataframes:")
for layer in TARGET_LAYERS:
    layer = int(layer)
    print(f"  active_freq_rank_df_layer_{layer}")


In [ ]:

# ============================================================
# 6. Per-target-marker feature rankings, per layer
# ============================================================


def rank_features_for_single_marker(layer: int, marker_number: int):
    layer = int(layer)
    marker_number = int(marker_number)
    layer_analysis = analysis_by_layer[layer]
    stats = layer_analysis["stats"]
    marker_numbers = stats["marker_numbers"]

    if marker_number not in set(marker_numbers.tolist()):
        raise ValueError(f"layer={layer}, marker_number={marker_number} not present. Available: {marker_numbers.tolist()}")

    single_mask = marker_numbers == marker_number
    other_mask = marker_numbers != marker_number

    df = build_group_contrast_df(
        stats,
        single_mask,
        other_mask,
        contrast_name=f"layer_{layer}_before_example_{marker_number}_vs_all_other_example_markers",
    )

    stronger = df.sort_values(
        ["welch_t_target_gt_other", "target_minus_other_mean", "cohen_d_target_vs_other"],
        ascending=False,
    ).reset_index(drop=True)

    weaker = df.sort_values(
        ["welch_t_target_gt_other", "target_minus_other_mean", "cohen_d_target_vs_other"],
        ascending=True,
    ).reset_index(drop=True)

    return df, stronger, weaker


per_marker_rankings_by_layer = {}
for layer in TARGET_LAYERS:
    layer = int(layer)
    per_marker_rankings_by_layer[layer] = {}
    layer_dir = os.path.join(OUTPUT_DIR, f"layer_{layer}")

    for marker_number in TARGET_MARKER_NUMBERS:
        df, stronger, weaker = rank_features_for_single_marker(layer, marker_number)
        per_marker_rankings_by_layer[layer][int(marker_number)] = {
            "df": df,
            "stronger": stronger,
            "weaker": weaker,
        }

        df.to_csv(os.path.join(layer_dir, f"before_example_{marker_number}_vs_other_contrast.csv"), index=False)
        stronger.to_csv(os.path.join(layer_dir, f"before_example_{marker_number}_stronger_rank_df.csv"), index=False)
        weaker.to_csv(os.path.join(layer_dir, f"before_example_{marker_number}_weaker_rank_df.csv"), index=False)

        print("=" * 110)
        print(f"Layer {layer} | Before Tool {marker_number} / after {marker_number - 1} completed tools: top STRONGER features")
        display(stronger.head(20))

print("Saved per-marker contrast CSVs to:", OUTPUT_DIR)


In [ ]:
import plotly.io as pio

# Renderer for the classic Jupyter Notebook interface.
pio.renderers.default = "notebook_connected"

# Positions = [-1]

In [ ]:
# ============================================================
# 7. Current plots for each layer: feature curves and heatmaps
# ============================================================


def position_offsets_label():
    return position_selection_label(POSITION_OFFSETS)


def marker_numbers_to_completed_examples(marker_numbers):
    """Display x-axis as completed tool count, so Before Tool 2 appears at x=1."""
    return np.asarray(marker_numbers, dtype=np.int64) - 1


def make_feature_marker_curve_df(layer: int, feature_indices):
    layer = int(layer)
    feature_indices = [int(f) for f in feature_indices]
    layer_analysis = analysis_by_layer[layer]
    stats = layer_analysis["stats"]
    marker_numbers = stats["marker_numbers"]
    completed_examples = marker_numbers_to_completed_examples(marker_numbers)
    mean_by_marker = stats["mean_by_marker"]
    active_freq_by_marker = stats["active_freq_by_marker"]
    count_by_marker = stats["counts"].astype(int)
    target_marker_set = set(int(x) for x in TARGET_MARKER_NUMBERS)

    rows = []
    for f in feature_indices:
        for i, marker_number in enumerate(marker_numbers):
            completed = int(completed_examples[i])
            rows.append({
                "layer": layer,
                "feature_idx": f,
                "marker_number": int(marker_number),
                "completed_examples": completed,
                "position_label": f"After {completed} completed tools / Before Tool {int(marker_number)}",
                "mean_activation": float(mean_by_marker[i, f]),
                "active_freq": float(active_freq_by_marker[i, f]),
                "is_target_marker": int(marker_number) in target_marker_set,
                "is_modulo10_completed_count": completed in set(int(x) for x in MODULO_TARGET_COMPLETED_COUNTS),
                "count": int(count_by_marker[i]),
            })
    return pd.DataFrame(rows)


def plot_feature_marker_curve(layer: int, feature_idx: int):
    layer = int(layer)
    feature_idx = int(feature_idx)
    df = make_feature_marker_curve_df(layer, [feature_idx])
    layer_analysis = analysis_by_layer[layer]
    combined_contrast_df = layer_analysis["combined_contrast_df"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["completed_examples"],
        y=df["mean_activation"],
        mode="lines+markers",
        name=f"layer {layer}, feature {feature_idx}",
        customdata=df[["marker_number"]].to_numpy(),
        hovertemplate="after %{x} completed tools<br>Before Tool %{customdata[0]}<br>mean activation=%{y:.5f}<extra></extra>",
    ))

    for completed_count in MODULO_TARGET_COMPLETED_COUNTS:
        fig.add_vline(x=completed_count, line_dash="dot", opacity=0.45)

    row = combined_contrast_df.loc[combined_contrast_df["feature_idx"] == feature_idx].iloc[0]
    fig.update_layout(
        title=(
            f"Layer {layer} SAE feature {feature_idx}: mean activation by completed-example count<br>"
            f"modulo10 target-minus-other={row['target_minus_other_mean']:.4f}, "
            f"Welch t={row['welch_t_target_gt_other']:.2f}, "
            f"Cohen d={row['cohen_d_target_vs_other']:.2f}"
        ),
        xaxis_title="Completed tool count (state before the next Tool marker)",
        yaxis_title=f"Mean {SAE_VALUE_LABEL} over positions [{position_offsets_label()}]",
        width=PAPER_FIG_WIDTH,
        height=PAPER_FIG_HEIGHT,
        template="simple_white",
    )
    show_figure(fig, filename="figure_cell_21_1")
    return df


def plot_top_feature_curves(layer: int, rank_df, top_k=12, title="Top target-position features"):
    layer = int(layer)
    feature_indices = rank_df["feature_idx"].head(top_k).astype(int).tolist()
    df = make_feature_marker_curve_df(layer, feature_indices)

    fig = px.line(
        df,
        x="completed_examples",
        y="mean_activation",
        color="feature_idx",
        markers=True,
        title=title,
        hover_data=["marker_number", "position_label", "active_freq", "is_target_marker", "count"],
    )
    for completed_count in MODULO_TARGET_COMPLETED_COUNTS:
        fig.add_vline(x=completed_count, line_dash="dot", opacity=0.25)
    fig.update_layout(
        width=PAPER_FIG_WIDTH,
        height=PAPER_FIG_HEIGHT,
        xaxis_title="Completed tool count (state before the next Tool marker)",
        yaxis_title=f"Mean {SAE_VALUE_LABEL} over positions [{position_offsets_label()}]",
        template="simple_white",
    )
    show_figure(fig, filename="figure_cell_21_2")
    return df


def row_zscore_feature_curves(mean_by_marker: np.ndarray, feature_indices):
    feature_indices = [int(f) for f in feature_indices]
    Z = mean_by_marker[:, feature_indices].T  # [features, markers]
    Z_z = (Z - Z.mean(axis=1, keepdims=True)) / (Z.std(axis=1, keepdims=True) + 1e-9)
    return Z_z


def plot_feature_marker_heatmap(layer: int, feature_indices, title):
    layer = int(layer)
    feature_indices = [int(f) for f in feature_indices]
    stats = analysis_by_layer[layer]["stats"]
    marker_numbers = stats["marker_numbers"]
    completed_examples = marker_numbers_to_completed_examples(marker_numbers)
    Z_z = row_zscore_feature_curves(stats["mean_by_marker"], feature_indices)

    fig = go.Figure(data=go.Heatmap(
        z=Z_z,
        x=completed_examples,
        y=[str(f) for f in feature_indices],
        colorscale="RdBu",
        zmid=0.0,
        colorbar=dict(
            title=dict(
                text="row z-scored<br>mean activation",
                font=dict(family="Times New Roman")
            ),
            tickfont=dict(family="Times New Roman")
        ),
        hovertemplate="feature %{y}<br>after %{x} completed examples<br>z=%{z:.3f}<extra></extra>",
    ))

    # Add vertical dashed lines
    for completed_count in MODULO_TARGET_COMPLETED_COUNTS:
        if completed_count in set(completed_examples.tolist()):
            fig.add_vline(x=completed_count, line_dash="dot", opacity=0.35)

    # Red downward triangles at columns 10, 20, 30, 40, 50.
    triangle_columns = [10, 20, 30, 40, 50]
    for x_val in triangle_columns:
        if x_val in set(completed_examples.tolist()):
            fig.add_annotation(
                x=x_val,
                y=1.03,                  # Coordinates relative to the plot area ceiling (1.0 = top)
                xref="x",               # Uses the data scale for x-axis alignment
                yref="paper",           # Uses layout scale for y-axis positioning
                text="▼",               # Unicode upside-down triangle
                showarrow=False,
                font=dict(color="red", size=24),
                yshift=10,              # Shifts 10 pixels up so it hovers cleanly right above the grid line
            )

    # Store calculated height for layout consistency and high-res scaling 
    calculated_height = max(600, 26 * len(feature_indices))

    fig.update_layout(
        font=dict(family="Times New Roman"), 
        title=title,
        title_font_size=26,                  
        xaxis_title="Exemplar Index",
        xaxis_title_font_size=22,             
        yaxis_title="SAE feature index",
        yaxis_title_font_size=22,             
        width=900,
        height=calculated_height,
        template="simple_white",
    )
    
    # Dynamic Configuration to boost download resolution scale by 4x
    config = {
        'toImageButtonOptions': {
            'format': 'png',
            'filename': f'feature_marker_heatmap_layer_{layer}',
            'height': calculated_height,
            'width': 900,
            'scale': 4 
        }
    }

    fig.show(config=config)
    return fig


PLOT_CURRENT_PLOTS_FOR_EACH_LAYER = True
TOP_K_CURVES = 12
TOP_K_HEATMAP = 30

if PLOT_CURRENT_PLOTS_FOR_EACH_LAYER:
    for layer in TARGET_LAYERS:
        layer = int(layer)
        print("=" * 120)
        print(f"Layer {layer}: current plots")
        print("=" * 120)
        stronger_df = analysis_by_layer[layer]["combined_stronger_rank_df"]
        weaker_df = analysis_by_layer[layer]["combined_weaker_rank_df"]

        top_stronger_feature = int(stronger_df.iloc[0]["feature_idx"])
        top_weaker_feature = int(weaker_df.iloc[0]["feature_idx"])

        print(f"Layer {layer} top combined stronger feature:", top_stronger_feature)
        _ = plot_feature_marker_curve(layer, top_stronger_feature)

        print(f"Layer {layer} top combined weaker feature:", top_weaker_feature)
        _ = plot_feature_marker_curve(layer, top_weaker_feature)

        _ = plot_top_feature_curves(
            layer,
            stronger_df,
            top_k=TOP_K_CURVES,
            title=f"Layer {layer}: Top {TOP_K_CURVES} features stronger at modulo-10 completed-count positions",
        )

        _ = plot_top_feature_curves(
            layer,
            weaker_df,
            top_k=TOP_K_CURVES,
            title=f"Layer {layer}: Top {TOP_K_CURVES} features weaker at modulo-10 completed-count positions",
        )

        top_stronger_features = stronger_df["feature_idx"].head(TOP_K_HEATMAP).astype(int).tolist()
        top_weaker_features = weaker_df["feature_idx"].head(TOP_K_HEATMAP).astype(int).tolist()

        plot_feature_marker_heatmap(
            layer,
            top_stronger_features,
            title=f"DPE Features (Task: BFCL, Model: Qwen3-14B-IT, Layer: {layer})",
        )

        plot_feature_marker_heatmap(
            layer,
            top_weaker_features,
            title=f"Layer {layer}: top weaker features, row-standardized mean activation across completed-example positions",
        )
else:
    print("Set PLOT_CURRENT_PLOTS_FOR_EACH_LAYER=True to render per-layer plots.")



## Layer-level metric for “stronger modulo-10 behavior”

A simple “average the top-10 or top-20 strongest modulo-10 features” metric is reasonable as a first diagnostic, but it is circular if the same data are used both to select the features and to estimate the layer’s strength. This notebook therefore uses a more defensible metric:

**Cross-validated top-K modulo-10 z-contrast.** For each layer, split prompts into two deterministic halves. On the selection split, compute each feature’s row-z-scored marker curve and rank features by:

```text
mean z at modulo-10 completed-count positions - mean z at non-modulo positions
```

Then evaluate the selected top-K features on the held-out split. The reported score averages both directions: select-even/evaluate-odd and select-odd/evaluate-even. This keeps the intuitive “top-K modulo features” idea, but reduces winner’s-curse inflation.

The most useful visualizations are:

1. a line plot of the held-out top-K score versus layer;
2. a layer-by-position heatmap of the held-out average z-profile of the selected features;
3. a distribution/box plot of all feature-level modulo z-contrast scores per layer.


In [ ]:

# ============================================================
# 8. Cross-validated modulo-10 layer metric and visualizations
# ============================================================


def feature_modulo_z_scores_from_stats(stats: Dict[str, Any], target_marker_numbers=TARGET_MARKER_NUMBERS):
    """
    Compute feature-level modulo-10 score from row-z-scored marker curves.

    score_f = mean_z_f(modulo10 target markers) - mean_z_f(other markers)
    where z-scoring is across marker positions within each feature.
    """
    marker_numbers = stats["marker_numbers"]
    mean_by_marker = stats["mean_by_marker"]  # [markers, features]
    Z = mean_by_marker.T  # [features, markers]
    Z_z = (Z - Z.mean(axis=1, keepdims=True)) / (Z.std(axis=1, keepdims=True) + 1e-9)

    target_set = set(int(x) for x in target_marker_numbers)
    target_mask = np.asarray([int(m) in target_set for m in marker_numbers], dtype=bool)
    other_mask = ~target_mask

    scores = Z_z[:, target_mask].mean(axis=1) - Z_z[:, other_mask].mean(axis=1)
    return scores, Z_z, target_mask


def topk_mean(scores, k: int):
    k = int(min(k, len(scores)))
    if k <= 0:
        return np.nan, np.asarray([], dtype=int)
    idx = np.argsort(scores)[::-1][:k]
    return float(np.mean(scores[idx])), idx.astype(int)


def compute_cv_modulo_metric_for_layer(layer: int, top_k: int = TOP_K_FOR_LAYER_METRIC):
    layer = int(layer)
    layer_output = layer_sweep_outputs[layer]

    stats_all = compute_marker_stats(layer_output, split_name="all")
    stats_select = compute_marker_stats(layer_output, split_name="select")
    stats_eval = compute_marker_stats(layer_output, split_name="eval")

    scores_all, Z_all, target_mask = feature_modulo_z_scores_from_stats(stats_all)
    scores_select, Z_select, _ = feature_modulo_z_scores_from_stats(stats_select)
    scores_eval, Z_eval, _ = feature_modulo_z_scores_from_stats(stats_eval)

    in_sample_topk_score, top_all = topk_mean(scores_all, top_k)

    # Direction 1: select on even prompts, evaluate on odd prompts.
    _, top_select = topk_mean(scores_select, top_k)
    heldout_select_to_eval = float(np.mean(scores_eval[top_select]))
    profile_select_to_eval = Z_eval[top_select].mean(axis=0)

    # Direction 2: select on odd prompts, evaluate on even prompts.
    _, top_eval = topk_mean(scores_eval, top_k)
    heldout_eval_to_select = float(np.mean(scores_select[top_eval]))
    profile_eval_to_select = Z_select[top_eval].mean(axis=0)

    cv_topk_score = float(np.mean([heldout_select_to_eval, heldout_eval_to_select]))
    cv_profile = 0.5 * (profile_select_to_eval + profile_eval_to_select)

    # Additional population-level diagnostics.
    positive_count = int(np.sum(scores_all > 0))
    positive_frac = float(np.mean(scores_all > 0))
    q95 = float(np.quantile(scores_all, 0.95))
    q99 = float(np.quantile(scores_all, 0.99))

    row = {
        "layer": layer,
        "top_k": int(top_k),
        "cv_topk_modulo_z_contrast": cv_topk_score,
        "heldout_select_to_eval": heldout_select_to_eval,
        "heldout_eval_to_select": heldout_eval_to_select,
        "in_sample_topk_modulo_z_contrast": in_sample_topk_score,
        "all_feature_mean_modulo_z_contrast": float(np.mean(scores_all)),
        "all_feature_median_modulo_z_contrast": float(np.median(scores_all)),
        "all_feature_q95_modulo_z_contrast": q95,
        "all_feature_q99_modulo_z_contrast": q99,
        "positive_feature_count": positive_count,
        "positive_feature_frac": positive_frac,
        "n_features": int(len(scores_all)),
        "top_features_all_split": ",".join(map(str, top_all.tolist())),
        "top_features_select_split": ",".join(map(str, top_select.tolist())),
        "top_features_eval_split": ",".join(map(str, top_eval.tolist())),
    }

    score_df = pd.DataFrame({
        "layer": layer,
        "feature_idx": np.arange(len(scores_all), dtype=int),
        "modulo_z_contrast_all": scores_all,
        "modulo_z_contrast_select": scores_select,
        "modulo_z_contrast_eval": scores_eval,
        "is_top_all_split": np.isin(np.arange(len(scores_all)), top_all),
        "is_top_select_split": np.isin(np.arange(len(scores_all)), top_select),
        "is_top_eval_split": np.isin(np.arange(len(scores_all)), top_eval),
    })

    profile_df = pd.DataFrame({
        "layer": layer,
        "marker_number": stats_all["marker_numbers"].astype(int),
        "completed_examples": (stats_all["marker_numbers"] - 1).astype(int),
        "cv_topk_heldout_mean_z_profile": cv_profile,
        "is_modulo10_target": target_mask,
    })

    return row, score_df, profile_df


positions_text = position_selection_label(POSITION_OFFSETS)

metric_rows = []
score_dfs = []
profile_dfs = []
metric_rows_by_k = []

for layer in TARGET_LAYERS:
    row, score_df, profile_df = compute_cv_modulo_metric_for_layer(int(layer), top_k=TOP_K_FOR_LAYER_METRIC)
    metric_rows.append(row)
    score_dfs.append(score_df)
    profile_dfs.append(profile_df)

    for k in LAYER_METRIC_TOP_K_OPTIONS:
        krow, _, _ = compute_cv_modulo_metric_for_layer(int(layer), top_k=int(k))
        metric_rows_by_k.append(krow)

modulo_layer_metric_df = pd.DataFrame(metric_rows).sort_values("layer")
modulo_feature_score_df = pd.concat(score_dfs, ignore_index=True)
modulo_layer_profile_df = pd.concat(profile_dfs, ignore_index=True)
modulo_layer_metric_by_k_df = pd.DataFrame(metric_rows_by_k).sort_values(["top_k", "layer"])

modulo_layer_metric_df.to_csv(os.path.join(OUTPUT_DIR, "modulo10_layer_metric_cv_topk.csv"), index=False)
modulo_feature_score_df.to_csv(os.path.join(OUTPUT_DIR, "modulo10_feature_scores_by_layer.csv"), index=False)
modulo_layer_profile_df.to_csv(os.path.join(OUTPUT_DIR, "modulo10_layer_position_profile_cv_topk.csv"), index=False)
modulo_layer_metric_by_k_df.to_csv(os.path.join(OUTPUT_DIR, "modulo10_layer_metric_cv_by_topk.csv"), index=False)

print("Cross-validated layer metric:")
display(modulo_layer_metric_df)
print("Cross-validated layer metric by top-k:")
display(modulo_layer_metric_by_k_df)

# 1. Main layer-strength plot.
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=modulo_layer_metric_df["layer"],
    y=modulo_layer_metric_df["cv_topk_modulo_z_contrast"],
    mode="lines+markers",
    name=f"held-out top-{TOP_K_FOR_LAYER_METRIC}",
    hovertemplate="layer %{x}<br>CV top-k score=%{y:.3f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=modulo_layer_metric_df["layer"],
    y=modulo_layer_metric_df["in_sample_topk_modulo_z_contrast"],
    mode="lines+markers",
    name=f"in-sample top-{TOP_K_FOR_LAYER_METRIC}",
    hovertemplate="layer %{x}<br>in-sample top-k score=%{y:.3f}<extra></extra>",
))
fig.update_layout(
    title=f"Modulo-10 layer strength: cross-validated top-{TOP_K_FOR_LAYER_METRIC} row-z contrast; positions=[{positions_text}]",
    xaxis_title="Layer index, post model.layers[layer] residual",
    yaxis_title="mean z(modulo-10 targets) - mean z(other positions)",
    width=PAPER_FIG_WIDTH,
    height=PAPER_FIG_HEIGHT,
    template="simple_white",
)
show_figure(fig, filename="figure_cell_24_1")

# 2. Top-k sensitivity plot.
fig = px.line(
    modulo_layer_metric_by_k_df,
    x="layer",
    y="cv_topk_modulo_z_contrast",
    color="top_k",
    markers=True,
    title=f"Modulo-10 layer strength across top-k choices; positions=[{positions_text}]",
    hover_data=["in_sample_topk_modulo_z_contrast", "positive_feature_count", "positive_feature_frac"],
)
fig.update_layout(
    xaxis_title="Layer index",
    yaxis_title="held-out top-k modulo z-contrast",
    width=PAPER_FIG_WIDTH,
    height=PAPER_FIG_HEIGHT,
    template="simple_white",
)
show_figure(fig, filename="figure_cell_24_2")

# 3. Layer-by-position held-out profile heatmap.
profile_pivot = modulo_layer_profile_df.pivot(index="layer", columns="completed_examples", values="cv_topk_heldout_mean_z_profile")
fig = go.Figure(data=go.Heatmap(
    z=profile_pivot.values,
    x=profile_pivot.columns.astype(int).tolist(),
    y=profile_pivot.index.astype(int).tolist(),
    colorscale="RdBu",
    zmid=0.0,
    colorbar=dict(title="held-out<br>mean z"),
    hovertemplate="layer %{y}<br>after %{x} completed tools<br>mean z=%{z:.3f}<extra></extra>",
))
for completed_count in MODULO_TARGET_COMPLETED_COUNTS:
    fig.add_vline(x=completed_count, line_dash="dot", opacity=0.30)
fig.update_layout(
    title=f"Held-out average z-profile of cross-validated top-{TOP_K_FOR_LAYER_METRIC} modulo features by completed-example count; positions=[{positions_text}]",
    xaxis_title="Completed tool count (state before the next Tool marker)",
    yaxis_title="Layer",
    width=PAPER_FIG_WIDTH,
    height=PAPER_FIG_HEIGHT,
    template="simple_white",
)
show_figure(fig, filename="figure_cell_24_3")

# 4. Distribution of feature-level modulo scores by layer.
plot_df = modulo_feature_score_df.copy()
plot_df["layer_str"] = plot_df["layer"].astype(str)
fig = px.box(
    plot_df,
    x="layer_str",
    y="modulo_z_contrast_all",
    points=False,
    title="Distribution of feature-level modulo-10 z-contrast scores by layer",
)
fig.update_layout(
    xaxis_title="Layer",
    yaxis_title="feature-level z(modulo targets) - z(other positions)",
    width=PAPER_FIG_WIDTH,
    height=PAPER_FIG_HEIGHT,
    template="simple_white",
)
show_figure(fig, filename="figure_cell_24_4")


In [ ]:

# ============================================================
# 9. Optional: inspect custom feature lists by layer
# ============================================================

# Example format:
# CUSTOM_FEATURES_TO_INSPECT_BY_LAYER = {
#     20: [123, 456],
#     40: [789],
# }
CUSTOM_FEATURES_TO_INSPECT_BY_LAYER = {}

for layer, features in CUSTOM_FEATURES_TO_INSPECT_BY_LAYER.items():
    layer = int(layer)
    if len(features) == 0:
        continue
    custom_df = plot_top_feature_curves(
        layer,
        pd.DataFrame({"feature_idx": [int(f) for f in features]}),
        top_k=len(features),
        title=f"Layer {layer}: custom feature marker-position curves",
    )
    plot_feature_marker_heatmap(
        layer,
        features,
        title=f"Layer {layer}: custom features, row-standardized marker-position heatmap",
    )
    display(custom_df.head())

if len(CUSTOM_FEATURES_TO_INSPECT_BY_LAYER) == 0:
    print("Set CUSTOM_FEATURES_TO_INSPECT_BY_LAYER to inspect specific layer/feature pairs.")


In [ ]:

# ============================================================
# 10. Compact summary tables across layers
# ============================================================

summary_rows = []
for layer in TARGET_LAYERS:
    layer = int(layer)
    for name, df in [
        ("combined_stronger", analysis_by_layer[layer]["combined_stronger_rank_df"]),
        ("combined_weaker", analysis_by_layer[layer]["combined_weaker_rank_df"]),
    ]:
        for rank, row in df.head(20).iterrows():
            summary_rows.append({
                "layer": layer,
                "group": name,
                "rank": int(rank + 1),
                "feature_idx": int(row["feature_idx"]),
                "target_minus_other_mean": float(row["target_minus_other_mean"]),
                "welch_t_target_gt_other": float(row["welch_t_target_gt_other"]),
                "cohen_d_target_vs_other": float(row["cohen_d_target_vs_other"]),
                "target_active_freq": float(row["target_active_freq"]),
                "other_active_freq": float(row["other_active_freq"]),
                "target_minus_other_active_freq": float(row["target_minus_other_active_freq"]),
            })

compact_summary_df = pd.DataFrame(summary_rows)
compact_summary_df.to_csv(os.path.join(OUTPUT_DIR, "compact_top20_stronger_weaker_summary_by_layer.csv"), index=False)
display(compact_summary_df)
print("Saved compact summary and metric outputs to:", OUTPUT_DIR)


## 11. DPE component analysis across selected features

Select layer-feature pairs using the modulo-10 statistics, then scan nearby attention heads and MLP paths for component writes that are both (i) stronger at modulo-10 boundary positions and (ii) aligned with the selected feature activation. The final cells provide an attention-vs-MLP comparison and optional causal intervention tests.


In [ ]:
# ============================================================
# 11A. Select DPE features for component analysis
# ============================================================

from typing import Sequence, Union
from scipy.stats import pearsonr

# -----------------------------
# User-editable config
# -----------------------------

# Number of layer-feature pairs to carry into component analysis.
DPE_COMPONENT_NUM_FEATURES = 8

# "positive" selects features higher at modulo-10 completed-count positions.
# "absolute" allows either higher or lower periodic features and stores the sign.
DPE_COMPONENT_FEATURE_SELECTION_MODE = "positive"  # "positive" or "absolute"

# Minimum held-out stability. Set to None to disable.
# For positive mode, this is min(select_score, eval_score).
# For absolute mode, this is min(abs(select_score), abs(eval_score)).
DPE_COMPONENT_MIN_STABLE_DPE_SCORE = None

# Tie-breaker for Priority 2: prefer features with stronger raw modulo-10 preactivation.
# Options: "target_mean", "target_minus_other_mean", "abs_target_minus_other_mean".
DPE_COMPONENT_PRIORITY2_COLUMN = "target_mean"


def _safe_pearson_np(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 3 or np.std(x) <= 1e-12 or np.std(y) <= 1e-12:
        return np.nan
    return float(pearsonr(x, y)[0])


def build_dpe_feature_candidate_table(
    *,
    layers=TARGET_LAYERS,
    selection_mode: str = DPE_COMPONENT_FEATURE_SELECTION_MODE,
    priority2_column: str = DPE_COMPONENT_PRIORITY2_COLUMN,
    min_stable_score=DPE_COMPONENT_MIN_STABLE_DPE_SCORE,
):
    """
    Build a layer-feature table for DPE component analysis.

    Priority 1: cross-split-stable modulo-10 z-contrast.
    Priority 2: raw modulo-target preactivation strength.

    A row corresponds to one SAE feature at one target SAE layer.
    """
    if selection_mode not in {"positive", "absolute"}:
        raise ValueError("selection_mode must be 'positive' or 'absolute'.")

    rows = []
    for layer in layers:
        layer = int(layer)
        layer_output = layer_sweep_outputs[layer]
        stats_all = compute_marker_stats(layer_output, split_name="all")
        stats_select = compute_marker_stats(layer_output, split_name="select")
        stats_eval = compute_marker_stats(layer_output, split_name="eval")

        scores_all, Z_all, target_mask = feature_modulo_z_scores_from_stats(stats_all)
        scores_select, _, _ = feature_modulo_z_scores_from_stats(stats_select)
        scores_eval, _, _ = feature_modulo_z_scores_from_stats(stats_eval)
        other_mask = ~target_mask

        mean_by_marker = stats_all["mean_by_marker"]
        active_by_marker = stats_all["active_freq_by_marker"]

        target_mean = mean_by_marker[target_mask].mean(axis=0)
        other_mean = mean_by_marker[other_mask].mean(axis=0)
        target_minus_other_mean = target_mean - other_mean
        target_active_freq = active_by_marker[target_mask].mean(axis=0)
        other_active_freq = active_by_marker[other_mask].mean(axis=0)

        if selection_mode == "positive":
            expected_sign = np.ones_like(scores_all, dtype=np.float64)
            stable_score = np.minimum(scores_select, scores_eval)
            priority1 = stable_score
        else:
            expected_sign = np.where(scores_all >= 0, 1.0, -1.0)
            stable_score = np.minimum(np.abs(scores_select), np.abs(scores_eval))
            priority1 = stable_score

        if min_stable_score is not None:
            keep = stable_score >= float(min_stable_score)
        else:
            keep = np.ones_like(stable_score, dtype=bool)

        if priority2_column == "target_mean":
            priority2 = target_mean
        elif priority2_column == "target_minus_other_mean":
            priority2 = target_minus_other_mean
        elif priority2_column == "abs_target_minus_other_mean":
            priority2 = np.abs(target_minus_other_mean)
        else:
            raise ValueError(f"Unknown priority2_column={priority2_column!r}")

        n_features = int(len(scores_all))
        for f in range(n_features):
            if not bool(keep[f]):
                continue
            rows.append({
                "target_layer": int(layer),
                "feature_idx": int(f),
                "expected_sign": float(expected_sign[f]),
                "modulo_z_contrast_all": float(scores_all[f]),
                "modulo_z_contrast_select": float(scores_select[f]),
                "modulo_z_contrast_eval": float(scores_eval[f]),
                "stable_dpe_score": float(stable_score[f]),
                "priority1_dpe_score": float(priority1[f]),
                "target_mean": float(target_mean[f]),
                "other_mean": float(other_mean[f]),
                "target_minus_other_mean": float(target_minus_other_mean[f]),
                "abs_target_minus_other_mean": float(abs(target_minus_other_mean[f])),
                "target_active_freq": float(target_active_freq[f]),
                "other_active_freq": float(other_active_freq[f]),
                "target_minus_other_active_freq": float(target_active_freq[f] - other_active_freq[f]),
                "priority2_score": float(priority2[f]),
            })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError("No DPE candidate features survived the current filters.")

    # Priority 1 dominates; Priority 2 breaks ties / near-ties.
    df = df.sort_values(
        ["priority1_dpe_score", "priority2_score", "abs_target_minus_other_mean"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    df["dpe_component_feature_rank"] = np.arange(1, len(df) + 1)
    return df


dpe_component_feature_candidate_df = build_dpe_feature_candidate_table()
dpe_selected_features_df = dpe_component_feature_candidate_df.head(int(DPE_COMPONENT_NUM_FEATURES)).copy()

DPE_SELECTED_FEATURES_FOR_COMPONENT_ANALYSIS = [
    {
        "target_layer": int(row.target_layer),
        "feature_idx": int(row.feature_idx),
        "expected_sign": float(row.expected_sign),
    }
    for row in dpe_selected_features_df.itertuples(index=False)
]

out_path = os.path.join(OUTPUT_DIR, "dpe_component_selected_features.csv")
dpe_component_feature_candidate_df.to_csv(os.path.join(OUTPUT_DIR, "dpe_component_all_feature_candidates.csv"), index=False)
dpe_selected_features_df.to_csv(out_path, index=False)

print("Selected DPE features for component analysis:")
display(dpe_selected_features_df[[
    "dpe_component_feature_rank", "target_layer", "feature_idx", "expected_sign",
    "stable_dpe_score", "modulo_z_contrast_all", "modulo_z_contrast_select", "modulo_z_contrast_eval",
    "target_mean", "other_mean", "target_minus_other_mean", "target_active_freq"
]])
print("Saved selected features to:", out_path)


In [ ]:
# ============================================================
# 11B. Component-write scan helpers for selected DPE features
# ============================================================

# -----------------------------
# User-editable config
# -----------------------------

RUN_DPE_COMPONENT_SCAN = True

# Scan the target layer and two preceding layers by default.
DPE_COMPONENT_RELATIVE_LAYERS = [0, -1, -2]

# Use fewer prompts for quick debugging; set to None to scan all 300 prompts.
DPE_COMPONENT_MAX_PROMPTS = 60

# Top candidates retained per feature for compact displays.
DPE_COMPONENT_TOP_N_PER_FEATURE = 12

# The component scan uses the same token selection as the main DPE analysis.
DPE_COMPONENT_POSITION_OFFSETS = POSITION_OFFSETS

EPS = 1e-9


def _dpe_unit(v: torch.Tensor) -> torch.Tensor:
    v = v.detach().float()
    return v / v.norm().clamp_min(1e-8)


def get_sae_read_write_dirs_any_provider(sae, feature_idx: int, device=None):
    """Return normalized SAE read and write directions for one feature.

    Supports both SAE Lens-style matrices:
      W_enc: [d_model, n_features], W_dec: [n_features, d_model]
    and direct-wrapper matrices:
      W_E: [d_model, n_features], W_D: [n_features, d_model].
    """
    feature_idx = int(feature_idx)

    # Decoder/write direction first, because this unambiguously gives d_model.
    if hasattr(sae, "W_dec"):
        W_dec = sae.W_dec.detach()
        if W_dec.shape[0] > feature_idx:
            write_dir = W_dec[feature_idx]
        elif W_dec.shape[1] > feature_idx:
            write_dir = W_dec[:, feature_idx]
        else:
            raise IndexError(f"feature_idx={feature_idx} is outside W_dec shape {tuple(W_dec.shape)}")
    elif hasattr(sae, "W_D"):
        W_D = sae.W_D.detach()
        if W_D.shape[0] > feature_idx:
            write_dir = W_D[feature_idx]
        elif W_D.shape[1] > feature_idx:
            write_dir = W_D[:, feature_idx]
        else:
            raise IndexError(f"feature_idx={feature_idx} is outside W_D shape {tuple(W_D.shape)}")
    else:
        raise AttributeError("Could not find SAE decoder/write matrix: expected W_dec or W_D.")

    d_model = int(write_dir.numel())

    # Encoder/read direction: choose the axis whose length matches d_model.
    if hasattr(sae, "W_enc"):
        W_enc = sae.W_enc.detach()
        if W_enc.shape[0] == d_model and W_enc.shape[1] > feature_idx:
            read_dir = W_enc[:, feature_idx]
        elif W_enc.shape[1] == d_model and W_enc.shape[0] > feature_idx:
            read_dir = W_enc[feature_idx]
        else:
            raise ValueError(
                f"Could not infer read direction for W_enc shape {tuple(W_enc.shape)} "
                f"and decoder d_model={d_model}."
            )
    elif hasattr(sae, "W_E"):
        W_E = sae.W_E.detach()
        if W_E.shape[0] == d_model and W_E.shape[1] > feature_idx:
            read_dir = W_E[:, feature_idx]
        elif W_E.shape[1] == d_model and W_E.shape[0] > feature_idx:
            read_dir = W_E[feature_idx]
        else:
            raise ValueError(
                f"Could not infer read direction for W_E shape {tuple(W_E.shape)} "
                f"and decoder d_model={d_model}."
            )
    else:
        raise AttributeError("Could not find SAE encoder/read matrix: expected W_enc or W_E.")

    if device is not None:
        read_dir = read_dir.to(device=device)
        write_dir = write_dir.to(device=device)
    return _dpe_unit(read_dir), _dpe_unit(write_dir)


def get_mlp_residual_write_module(model, layer_idx: int):
    """
    Return the module whose output is closest to the MLP residual-path write.
    For Llama this is normally block.mlp. For Gemma-style blocks this may be post_feedforward_layernorm.
    """
    block = model.get_submodule(f"model.layers.{int(layer_idx)}")
    for name in ["post_feedforward_layernorm", "post_mlp_layernorm", "post_ffn_layernorm"]:
        if hasattr(block, name):
            return getattr(block, name), name
    if hasattr(block, "mlp"):
        return block.mlp, "mlp"
    raise AttributeError(f"Could not find an MLP residual-write module for layer {layer_idx}.")


def build_dpe_marker_position_matrix_for_prompt(
    prompt: str,
    *,
    position_offsets=DPE_COMPONENT_POSITION_OFFSETS,
    skip_first_marker: bool = True,
):
    """Tokenize a prompt and return marker metadata plus flat measurement-position indexing."""
    input_device = get_input_device(model)
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        return_offsets_mapping=True,
        add_special_tokens=False,
    )
    offset_mapping = enc.pop("offset_mapping")[0].tolist()
    input_ids_cpu = enc["input_ids"][0].tolist()

    marker_info = get_example_marker_prediction_positions(
        prompt=prompt,
        input_ids_cpu=input_ids_cpu,
        offset_mapping=offset_mapping,
    )
    if skip_first_marker:
        marker_info = marker_info[1:]

    marker_info, position_rows = build_measurement_position_rows(
        marker_info=marker_info,
        input_ids_cpu=input_ids_cpu,
        position_offsets=position_offsets,
    )
    position_index = build_position_row_index(position_rows)

    enc["input_ids"] = enc["input_ids"].to(input_device)
    enc["attention_mask"] = enc["attention_mask"].to(input_device)
    return enc, marker_info, position_index


def marker_target_mask_from_info(marker_info):
    target_counts = set(int(x) for x in MODULO_TARGET_COMPLETED_COUNTS)
    return np.asarray([int(info["completed_examples"]) in target_counts for info in marker_info], dtype=bool)


def component_layers_for_target_layer(target_layer: int, relative_layers=DPE_COMPONENT_RELATIVE_LAYERS):
    layers = []
    for rel in relative_layers:
        layer = int(target_layer) + int(rel)
        if layer < 0:
            continue
        if layer >= int(getattr(model.config, "num_hidden_layers", layer + 1)):
            continue
        if layer not in layers:
            layers.append(layer)
    return layers


def dpe_promptwise_z_contrast(values, prompt_ids, target_mask):
    """Mean over prompts of z(target markers) - z(other markers)."""
    values = np.asarray(values, dtype=np.float64)
    prompt_ids = np.asarray(prompt_ids)
    target_mask = np.asarray(target_mask, dtype=bool)
    out = []
    for p in np.unique(prompt_ids):
        m = prompt_ids == p
        v = values[m]
        t = target_mask[m]
        finite = np.isfinite(v)
        if finite.sum() < 3 or t.sum() == 0 or (~t).sum() == 0:
            continue
        v = v[finite]
        t = t[finite]
        sd = float(np.std(v))
        if sd <= EPS:
            continue
        z = (v - float(np.mean(v))) / (sd + EPS)
        out.append(float(np.mean(z[t]) - np.mean(z[~t])))
    return float(np.nanmean(out)) if len(out) else np.nan


def dpe_point_biserial(values, target_mask):
    return _safe_pearson_np(np.asarray(values, dtype=np.float64), np.asarray(target_mask, dtype=np.float64))


def summarize_dpe_feature_values(feature_values, prompt_ids, target_mask, expected_sign=1.0):
    feature_values = np.asarray(feature_values, dtype=np.float64)
    target_mask = np.asarray(target_mask, dtype=bool)
    target_mean = float(np.nanmean(feature_values[target_mask]))
    other_mean = float(np.nanmean(feature_values[~target_mask]))
    contrast = dpe_promptwise_z_contrast(feature_values, prompt_ids, target_mask)
    r = dpe_point_biserial(feature_values, target_mask)
    return {
        "feature_target_mean": target_mean,
        "feature_other_mean": other_mean,
        "feature_target_minus_other_mean": target_mean - other_mean,
        "feature_promptwise_modulo_z_contrast": contrast,
        "feature_point_biserial_r_target_marker": r,
        "signed_feature_promptwise_modulo_z_contrast": float(expected_sign) * contrast,
        "signed_feature_point_biserial_r_target_marker": float(expected_sign) * r,
    }


def _current_feature_value_mode_for_provider():
    if SAE_PROVIDER == "qwen_batchtopk_direct":
        return QWEN_FEATURE_VALUE_MODE
    if SAE_PROVIDER in {"saelens", "gemma_scope_saelens"}:
        return SAELENS_FEATURE_VALUE_MODE
    if SAE_PROVIDER == "llama_scope_direct":
        return LLAMA_SCOPE_FEATURE_VALUE_MODE
    return "activation"


@torch.no_grad()
def sae_single_feature_flat_values_any_provider(sae, x_flat: torch.Tensor, feature_idx: int):
    """Memory-light single-feature values for the configured SAE feature-value mode."""
    feature_idx = int(feature_idx)
    sae_device = get_sae_device(sae, fallback=x_flat.device)
    sae_dtype = get_sae_dtype(sae, fallback=SAE_DTYPE)
    x = x_flat.to(device=sae_device, dtype=sae_dtype)
    mode = _current_feature_value_mode_for_provider()

    # Fast path for pre-activations.
    if mode == "preactivation":
        if hasattr(sae, "process_sae_in"):
            x_in = sae.process_sae_in(x)
        elif hasattr(sae, "b_dec"):
            x_in = x - sae.b_dec.to(device=x.device, dtype=x.dtype)
        else:
            x_in = x

        # Infer encoder orientation using the decoder/write dimension.
        if hasattr(sae, "W_dec"):
            W_dec = sae.W_dec.detach()
            d_model = int(W_dec.shape[1] if W_dec.shape[0] > feature_idx else W_dec.shape[0])
        elif hasattr(sae, "W_D"):
            W_D = sae.W_D.detach()
            d_model = int(W_D.shape[1] if W_D.shape[0] > feature_idx else W_D.shape[0])
        else:
            d_model = int(x_in.shape[-1])

        if hasattr(sae, "W_enc"):
            W = sae.W_enc.detach().to(device=x.device, dtype=x.dtype)
            if W.shape[0] == d_model and W.shape[1] > feature_idx:
                w = W[:, feature_idx]
            elif W.shape[1] == d_model and W.shape[0] > feature_idx:
                w = W[feature_idx]
            else:
                raise ValueError(f"Could not infer W_enc orientation from shape {tuple(W.shape)} and d_model={d_model}")
            b = sae.b_enc[feature_idx].to(device=x.device, dtype=x.dtype) if hasattr(sae, "b_enc") else 0.0
            return (x_in @ w + b).detach().float()

        if hasattr(sae, "W_E"):
            W = sae.W_E.detach().to(device=x.device, dtype=x.dtype)
            if W.shape[0] == d_model and W.shape[1] > feature_idx:
                w = W[:, feature_idx]
            elif W.shape[1] == d_model and W.shape[0] > feature_idx:
                w = W[feature_idx]
            else:
                raise ValueError(f"Could not infer W_E orientation from shape {tuple(W.shape)} and d_model={d_model}")
            b = sae.b_E[feature_idx].to(device=x.device, dtype=x.dtype) if hasattr(sae, "b_E") else 0.0
            vals = x_in @ w + b
            if getattr(sae, "sparsity_include_decoder_norm", False) and hasattr(sae, "W_D"):
                vals = vals * torch.norm(sae.W_D[feature_idx].detach().to(device=x.device, dtype=x.dtype), p=2).clamp_min(1e-8)
            return vals.detach().float()

    # Fallback: compute configured all-feature tensor and select one feature.
    feats, _ = _encode_sae_feature_tensors_flat(sae, x)
    return feats[:, feature_idx].detach().float()




def mean_flat_values_by_marker(values_flat: torch.Tensor, position_index):
    """Average flat per-token values over marker rows using position_index metadata."""
    values_flat = values_flat.float()
    n_markers = int(position_index["n_markers"])
    row_ids = torch.as_tensor(position_index["row_ids"], device=values_flat.device, dtype=torch.long)
    row_lengths = torch.as_tensor(position_index["row_lengths"], device=values_flat.device, dtype=values_flat.dtype).clamp_min(1)
    out_shape = (n_markers,) + tuple(values_flat.shape[1:])
    out = torch.zeros(out_shape, device=values_flat.device, dtype=values_flat.dtype)
    out.index_add_(0, row_ids, values_flat)
    denom = row_lengths
    while denom.ndim < out.ndim:
        denom = denom.unsqueeze(-1)
    return out / denom

@torch.no_grad()
def sae_single_feature_values_for_selected_hidden(sae, selected_hidden, feature_idx: int):
    """Single-feature values averaged over selected positions for each marker."""
    x_flat = selected_hidden_tensor(selected_hidden)
    vals_flat = sae_single_feature_flat_values_any_provider(sae, x_flat, feature_idx=feature_idx)
    if is_ragged_selected_hidden(selected_hidden):
        pseudo_index = {
            "n_markers": int(selected_hidden["n_markers"]),
            "row_ids": selected_hidden["row_ids"].detach().cpu().numpy(),
            "row_lengths": selected_hidden["row_lengths"].detach().cpu().numpy(),
        }
        vals_by_marker = mean_flat_values_by_marker(vals_flat, pseudo_index)
    else:
        n_markers = int(selected_hidden.shape[0])
        n_offsets = int(selected_hidden.shape[1]) if selected_hidden.ndim == 3 else 1
        vals_by_marker = vals_flat.reshape(n_markers, n_offsets).mean(dim=1)
    return vals_by_marker.detach().float().cpu().numpy()


@torch.no_grad()
def capture_dpe_feature_and_component_writes_for_prompt(
    prompt: str,
    *,
    target_sae,
    target_layer: int,
    feature_idx: int,
    component_layers: Sequence[int],
    position_offsets=DPE_COMPONENT_POSITION_OFFSETS,
):
    """
    One forward pass for one prompt. Captures:
      - target-layer hidden states at DPE measurement positions;
      - attention-head writes from component layers at the same positions;
      - MLP writes from component layers at the same positions.

    Returned component values are already averaged over the selected token positions for each marker.
    """
    target_layer = int(target_layer)
    feature_idx = int(feature_idx)
    component_layers = [int(x) for x in component_layers]

    enc, marker_info, position_index = build_dpe_marker_position_matrix_for_prompt(
        prompt,
        position_offsets=position_offsets,
        skip_first_marker=True,
    )
    flat_positions = position_index["flat_positions"].astype(int).tolist()
    if len(flat_positions) == 0:
        raise ValueError("No measurement positions were constructed for this prompt.")

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    cache = {
        "target_hidden_flat": None,
        "attn_o_proj_input": {},
        "mlp_output": {},
    }
    handles = []

    def target_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        idx = torch.tensor(flat_positions, device=x.device, dtype=torch.long)
        cache["target_hidden_flat"] = x[0, idx, :].detach().contiguous()
        return output

    def make_o_proj_pre_hook(layer_i: int):
        def hook(module, inputs):
            x = inputs[0]
            idx = torch.tensor(flat_positions, device=x.device, dtype=torch.long)
            cache["attn_o_proj_input"][int(layer_i)] = x[0, idx, :].detach().contiguous()
            return None
        return hook

    def make_mlp_hook(layer_i: int):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            idx = torch.tensor(flat_positions, device=x.device, dtype=torch.long)
            cache["mlp_output"][int(layer_i)] = x[0, idx, :].detach().contiguous()
            return output
        return hook

    try:
        # Target feature capture.
        target_block = model.get_submodule(f"model.layers.{target_layer}")
        handles.append(target_block.register_forward_hook(target_hook))

        # Component captures.
        for layer_i in component_layers:
            o_proj = model.get_submodule(f"model.layers.{layer_i}.self_attn.o_proj")
            handles.append(o_proj.register_forward_pre_hook(make_o_proj_pre_hook(layer_i)))

            mlp_module, _ = get_mlp_residual_write_module(model, layer_i)
            handles.append(mlp_module.register_forward_hook(make_mlp_hook(layer_i)))

        _ = get_backbone_model(model)(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        for h in handles:
            h.remove()

    if cache["target_hidden_flat"] is None:
        raise RuntimeError(f"Failed to capture target hidden states for layer {target_layer}.")

    selected_hidden = make_selected_hidden_from_flat(cache["target_hidden_flat"], position_index)
    target_feature_values = sae_single_feature_values_for_selected_hidden(
        target_sae,
        selected_hidden,
        feature_idx=feature_idx,
    )

    read_dir, write_dir = get_sae_read_write_dirs_any_provider(
        target_sae,
        feature_idx=feature_idx,
        device=cache["target_hidden_flat"].device,
    )

    component_values = {}
    for layer_i in component_layers:
        # MLP component: one write vector per selected token.
        mlp_flat = cache["mlp_output"][layer_i].float()
        mlp_read_flat = mlp_flat @ read_dir.to(device=mlp_flat.device, dtype=mlp_flat.dtype)
        mlp_write_flat = mlp_flat @ write_dir.to(device=mlp_flat.device, dtype=mlp_flat.dtype)
        component_values[("mlp", int(layer_i), -1, "read")] = mean_flat_values_by_marker(mlp_read_flat, position_index).detach().cpu().numpy()
        component_values[("mlp", int(layer_i), -1, "write")] = mean_flat_values_by_marker(mlp_write_flat, position_index).detach().cpu().numpy()

        # Attention heads: decompose o_proj input by head and use W_O projection vectors.
        attn = model.get_submodule(f"model.layers.{layer_i}.self_attn")
        o_proj = attn.o_proj
        W_o = o_proj.weight.detach().float()
        out_dim, in_dim = W_o.shape
        num_heads = int(getattr(attn, "num_heads", getattr(model.config, "num_attention_heads")))
        head_dim = int(getattr(attn, "head_dim", in_dim // num_heads))
        if in_dim != num_heads * head_dim:
            raise ValueError(f"Layer {layer_i}: o_proj input dim {in_dim} != num_heads*head_dim {num_heads}*{head_dim}")
        if read_dir.numel() != out_dim or write_dir.numel() != out_dim:
            raise ValueError(f"SAE direction dim does not match o_proj output dim at layer {layer_i}.")

        W_heads = W_o.reshape(out_dim, num_heads, head_dim).permute(1, 2, 0).contiguous()
        head_vec_read = torch.matmul(W_heads.to(read_dir.device), read_dir.float())
        head_vec_write = torch.matmul(W_heads.to(write_dir.device), write_dir.float())

        z_flat = cache["attn_o_proj_input"][layer_i].float()
        z_heads = z_flat.reshape(z_flat.shape[0], num_heads, head_dim)
        read_vals_flat = torch.einsum("thd,hd->th", z_heads, head_vec_read.to(z_heads.device).float())
        write_vals_flat = torch.einsum("thd,hd->th", z_heads, head_vec_write.to(z_heads.device).float())
        read_vals = mean_flat_values_by_marker(read_vals_flat, position_index).detach().cpu().numpy()
        write_vals = mean_flat_values_by_marker(write_vals_flat, position_index).detach().cpu().numpy()
        for h in range(num_heads):
            component_values[("attention_head", int(layer_i), int(h), "read")] = read_vals[:, h]
            component_values[("attention_head", int(layer_i), int(h), "write")] = write_vals[:, h]

    return {
        "marker_info": marker_info,
        "target_mask": marker_target_mask_from_info(marker_info),
        "target_feature_values": np.asarray(target_feature_values, dtype=np.float64),
        "component_values": component_values,
    }


In [ ]:
# ============================================================
# 11C. Run component-write scan for selected DPE features
# ============================================================


def run_dpe_component_scan_for_feature(
    *,
    target_layer: int,
    feature_idx: int,
    expected_sign: float,
    prompts=original_prompts,
    max_prompts=DPE_COMPONENT_MAX_PROMPTS,
    relative_layers=DPE_COMPONENT_RELATIVE_LAYERS,
    position_offsets=DPE_COMPONENT_POSITION_OFFSETS,
    verbose_every: int = 10,
):
    """Scan attention heads and MLP writes for one selected DPE feature."""
    target_layer = int(target_layer)
    feature_idx = int(feature_idx)
    expected_sign = float(expected_sign)
    component_layers = component_layers_for_target_layer(target_layer, relative_layers=relative_layers)
    prompts_to_run = prompts[:max_prompts] if max_prompts is not None else prompts

    target_sae = load_sae_for_layer(target_layer)

    feature_values_by_prompt = []
    target_masks_by_prompt = []
    prompt_ids_by_prompt = []
    component_values_by_key = {}

    print("=" * 120)
    print(f"DPE component scan: target layer={target_layer}, feature={feature_idx}, expected_sign={expected_sign:+.0f}")
    print("component layers:", component_layers)
    print("n prompts:", len(prompts_to_run))
    print("position selection:", position_selection_label(position_offsets))
    print("=" * 120)

    for prompt_idx, prompt in enumerate(prompts_to_run):
        measured = capture_dpe_feature_and_component_writes_for_prompt(
            prompt,
            target_sae=target_sae,
            target_layer=target_layer,
            feature_idx=feature_idx,
            component_layers=component_layers,
            position_offsets=position_offsets,
        )

        vals = measured["target_feature_values"]
        mask = measured["target_mask"]
        n_markers = int(len(vals))
        feature_values_by_prompt.append(vals)
        target_masks_by_prompt.append(mask)
        prompt_ids_by_prompt.append(np.full(n_markers, int(prompt_idx), dtype=np.int64))

        for key, arr in measured["component_values"].items():
            component_values_by_key.setdefault(key, []).append(np.asarray(arr, dtype=np.float64))

        if verbose_every is not None and (prompt_idx + 1) % int(verbose_every) == 0:
            print(f"processed {prompt_idx + 1}/{len(prompts_to_run)} prompts for feature {feature_idx}")

    feature_values = np.concatenate(feature_values_by_prompt, axis=0)
    target_mask = np.concatenate(target_masks_by_prompt, axis=0)
    prompt_ids = np.concatenate(prompt_ids_by_prompt, axis=0)

    feature_summary = summarize_dpe_feature_values(
        feature_values,
        prompt_ids,
        target_mask,
        expected_sign=expected_sign,
    )

    rows = []
    for key, arrs in component_values_by_key.items():
        component_type, component_layer, component_id, direction = key
        comp_values = np.concatenate(arrs, axis=0)
        comp_target_mean = float(np.nanmean(comp_values[target_mask]))
        comp_other_mean = float(np.nanmean(comp_values[~target_mask]))
        comp_target_minus_other = comp_target_mean - comp_other_mean
        comp_promptwise_contrast = dpe_promptwise_z_contrast(comp_values, prompt_ids, target_mask)
        comp_point_biserial = dpe_point_biserial(comp_values, target_mask)
        comp_feature_corr = _safe_pearson_np(comp_values, feature_values)

        signed_promptwise = expected_sign * comp_promptwise_contrast
        signed_point_biserial = expected_sign * comp_point_biserial
        # A compact candidate score: correct periodic direction, plus alignment with the target feature.
        dpe_write_score = (
            max(float(signed_promptwise), 0.0)
            * (0.5 + 0.5 * max(float(signed_point_biserial), 0.0))
            * (0.5 + 0.5 * max(float(comp_feature_corr), 0.0))
        )

        rows.append({
            "target_layer": int(target_layer),
            "feature_idx": int(feature_idx),
            "expected_sign": float(expected_sign),
            "component_type": component_type,
            "component_layer": int(component_layer),
            "component_id": int(component_id),
            "direction": direction,
            "component_target_mean": comp_target_mean,
            "component_other_mean": comp_other_mean,
            "component_target_minus_other_mean": comp_target_minus_other,
            "component_promptwise_modulo_z_contrast": comp_promptwise_contrast,
            "component_point_biserial_r_target_marker": comp_point_biserial,
            "component_write_target_feature_corr": comp_feature_corr,
            "signed_component_promptwise_modulo_z_contrast": signed_promptwise,
            "signed_component_point_biserial_r_target_marker": signed_point_biserial,
            "dpe_write_score": float(dpe_write_score),
            "n_events": int(len(comp_values)),
            "n_prompts": int(len(prompts_to_run)),
            **feature_summary,
        })

    summary_df = pd.DataFrame(rows).sort_values("dpe_write_score", ascending=False).reset_index(drop=True)
    summary_df["component_rank_for_feature"] = np.arange(1, len(summary_df) + 1)

    del target_sae
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary_df


def run_dpe_component_scan_for_selected_features(selected_feature_specs=DPE_SELECTED_FEATURES_FOR_COMPONENT_ANALYSIS):
    all_dfs = []
    for spec_idx, spec in enumerate(selected_feature_specs, start=1):
        print(f"\n\n### Component scan {spec_idx}/{len(selected_feature_specs)}: {spec}")
        df = run_dpe_component_scan_for_feature(
            target_layer=int(spec["target_layer"]),
            feature_idx=int(spec["feature_idx"]),
            expected_sign=float(spec.get("expected_sign", 1.0)),
        )
        all_dfs.append(df)
    out = pd.concat(all_dfs, ignore_index=True)
    out = out.sort_values(["target_layer", "feature_idx", "dpe_write_score"], ascending=[True, True, False]).reset_index(drop=True)
    return out


if RUN_DPE_COMPONENT_SCAN:
    dpe_component_summary_df = run_dpe_component_scan_for_selected_features()
    component_out_path = os.path.join(OUTPUT_DIR, "dpe_component_summary_selected_features.csv")
    dpe_component_summary_df.to_csv(component_out_path, index=False)
    print("Saved DPE component summary:", component_out_path)
    display(dpe_component_summary_df.head(50))
else:
    print("RUN_DPE_COMPONENT_SCAN=False. Set it to True to run the component scan.")


In [ ]:
# ============================================================
# 11D. MLP vs attention summaries and specific candidate components
# ============================================================


def _component_label(row):
    if row["component_type"] == "attention_head":
        return f"L{int(row['component_layer'])}H{int(row['component_id'])}-{row['direction']}"
    return f"L{int(row['component_layer'])} MLP-{row['direction']}"


def build_dpe_feature_component_comparison(component_summary_df: pd.DataFrame):
    """For each selected feature, compare strongest MLP candidate vs strongest attention-head candidate."""
    rows = []
    group_cols = ["target_layer", "feature_idx", "expected_sign"]
    for (target_layer, feature_idx, expected_sign), g in component_summary_df.groupby(group_cols):
        g = g.copy()
        g["component_label"] = g.apply(_component_label, axis=1)
        attn = g[g["component_type"] == "attention_head"].sort_values("dpe_write_score", ascending=False)
        mlp = g[g["component_type"] == "mlp"].sort_values("dpe_write_score", ascending=False)

        best_attn = attn.iloc[0] if len(attn) else None
        best_mlp = mlp.iloc[0] if len(mlp) else None
        best_attn_score = float(best_attn["dpe_write_score"]) if best_attn is not None else np.nan
        best_mlp_score = float(best_mlp["dpe_write_score"]) if best_mlp is not None else np.nan

        if np.isfinite(best_attn_score) and np.isfinite(best_mlp_score):
            winner = "MLP" if best_mlp_score > best_attn_score else "Attention"
            ratio = best_mlp_score / (best_attn_score + EPS)
        elif np.isfinite(best_mlp_score):
            winner = "MLP"
            ratio = np.inf
        elif np.isfinite(best_attn_score):
            winner = "Attention"
            ratio = 0.0
        else:
            winner = "none"
            ratio = np.nan

        # Rank read and write directions separately to avoid mixing SAE directions.
        best_read = g[g["direction"] == "read"].sort_values("dpe_write_score", ascending=False).head(1)
        best_write = g[g["direction"] == "write"].sort_values("dpe_write_score", ascending=False).head(1)

        rows.append({
            "target_layer": int(target_layer),
            "feature_idx": int(feature_idx),
            "expected_sign": float(expected_sign),
            "feature_signed_modulo_z_contrast": float(g["signed_feature_promptwise_modulo_z_contrast"].iloc[0]),
            "feature_target_minus_other_mean": float(g["feature_target_minus_other_mean"].iloc[0]),
            "best_attention_component": None if best_attn is None else best_attn["component_label"],
            "best_attention_score": best_attn_score,
            "best_mlp_component": None if best_mlp is None else best_mlp["component_label"],
            "best_mlp_score": best_mlp_score,
            "mlp_to_attention_score_ratio": float(ratio) if np.isfinite(ratio) else ratio,
            "stronger_component_type": winner,
            "best_read_component": None if len(best_read) == 0 else best_read.iloc[0]["component_label"],
            "best_read_score": np.nan if len(best_read) == 0 else float(best_read.iloc[0]["dpe_write_score"]),
            "best_write_component": None if len(best_write) == 0 else best_write.iloc[0]["component_label"],
            "best_write_score": np.nan if len(best_write) == 0 else float(best_write.iloc[0]["dpe_write_score"]),
        })
    return pd.DataFrame(rows).sort_values("feature_signed_modulo_z_contrast", ascending=False).reset_index(drop=True)


def build_dpe_top_component_table(component_summary_df: pd.DataFrame, top_n_per_feature: int = DPE_COMPONENT_TOP_N_PER_FEATURE):
    rows = []
    for (target_layer, feature_idx), g in component_summary_df.groupby(["target_layer", "feature_idx"]):
        g = g.copy().sort_values("dpe_write_score", ascending=False).head(int(top_n_per_feature))
        g["component_label"] = g.apply(_component_label, axis=1)
        rows.append(g)
    return pd.concat(rows, ignore_index=True).reset_index(drop=True)


if "dpe_component_summary_df" in globals() and len(dpe_component_summary_df) > 0:
    dpe_feature_component_comparison_df = build_dpe_feature_component_comparison(dpe_component_summary_df)
    dpe_top_component_candidates_df = build_dpe_top_component_table(dpe_component_summary_df)

    dpe_feature_component_comparison_df.to_csv(os.path.join(OUTPUT_DIR, "dpe_q1_mlp_vs_attention_by_feature.csv"), index=False)
    dpe_top_component_candidates_df.to_csv(os.path.join(OUTPUT_DIR, "dpe_q2_top_component_candidates_by_feature.csv"), index=False)

    print("Q1: MLP vs attention summary by feature")
    display(dpe_feature_component_comparison_df)

    print("Q2: Specific top component candidates by feature")
    display(dpe_top_component_candidates_df[[
        "target_layer", "feature_idx", "component_label", "component_type", "component_layer", "component_id", "direction",
        "dpe_write_score", "signed_component_promptwise_modulo_z_contrast",
        "component_write_target_feature_corr", "signed_component_point_biserial_r_target_marker",
        "feature_signed_modulo_z_contrast" if "feature_signed_modulo_z_contrast" in dpe_top_component_candidates_df.columns else "signed_feature_promptwise_modulo_z_contrast",
    ]].head(100))

    # Plot: best MLP vs best attention score per feature.
    plot_df = dpe_feature_component_comparison_df.copy()
    plot_df["feature_label"] = plot_df.apply(lambda r: f"L{int(r.target_layer)} F{int(r.feature_idx)}", axis=1)
    long_df = plot_df.melt(
        id_vars=["feature_label", "target_layer", "feature_idx"],
        value_vars=["best_attention_score", "best_mlp_score"],
        var_name="component_family",
        value_name="best_dpe_write_score",
    )
    long_df["component_family"] = long_df["component_family"].map({
        "best_attention_score": "Attention heads",
        "best_mlp_score": "MLPs",
    })
    fig = px.bar(
        long_df,
        x="feature_label",
        y="best_dpe_write_score",
        color="component_family",
        barmode="group",
        title="Best DPE-write component score by feature: attention heads vs MLPs",
        hover_data=["target_layer", "feature_idx"],
    )
    fig.update_layout(
        width=900,
        height=600,
        template="simple_white",
        font=dict(family="Times New Roman"),
        title=dict(font=dict(size=26)),
        xaxis=dict(title=dict(text="Selected DPE feature", font=dict(size=22)), tickfont=dict(size=14)),
        yaxis=dict(title=dict(text="Best DPE-write score", font=dict(size=22)), tickfont=dict(size=16)),
        legend_title_text="Component family",
    )
    fig.show(config={"toImageButtonOptions": {"format": "png", "filename": "dpe_q1_attention_vs_mlp", "width": 900, "height": 600, "scale": 4}})

    # Simple scatter: component periodicity vs feature alignment.
    scatter_df = dpe_component_summary_df.copy()
    scatter_df["component_family"] = scatter_df["component_type"].map({"attention_head": "Attention head", "mlp": "MLP"})
    scatter_df["component_label"] = scatter_df.apply(_component_label, axis=1)
    fig = px.scatter(
        scatter_df,
        x="signed_component_promptwise_modulo_z_contrast",
        y="component_write_target_feature_corr",
        color="component_family",
        symbol="direction",
        hover_data=["target_layer", "feature_idx", "component_label", "dpe_write_score"],
        title="DPE component-write geometry across selected features",
    )
    fig.add_vline(x=0, line_dash="dash", opacity=0.35)
    fig.add_hline(y=0, line_dash="dash", opacity=0.35)
    fig.update_layout(
        width=900,
        height=600,
        template="simple_white",
        font=dict(family="Times New Roman"),
        title=dict(font=dict(size=26)),
        xaxis=dict(title=dict(text="Component write periodicity", font=dict(size=22)), tickfont=dict(size=16)),
        yaxis=dict(title=dict(text="Correlation with DPE feature activation", font=dict(size=22)), tickfont=dict(size=16)),
        legend_title_text="Component",
    )
    fig.show(config={"toImageButtonOptions": {"format": "png", "filename": "dpe_q2_component_write_geometry", "width": 900, "height": 600, "scale": 4}})
else:
    print("Run the component scan first to create dpe_component_summary_df.")


In [ ]:
# ============================================================
# 11E. Causal intervention on candidate DPE components
# ============================================================

# The causal test scales a selected component's write at chosen marker positions and remeasures
# the selected DPE feature at the target SAE layer.
#
# alpha = 0   baseline
# alpha = -1  ablation, effective multiplier 0
# alpha > 0   amplification, effective multiplier 1 + alpha

RUN_DPE_COMPONENT_CAUSAL = False  # Set True to run.

DPE_CAUSAL_ALPHAS = [-5.0, -1.0, 0.0, 1.0, 5.0]
DPE_CAUSAL_MAX_PROMPTS = 40

# Recommended primary condition: intervene at all measured marker positions.
# This tests whether the same component scaling changes the modulo contrast more at modulo-10 positions.
# Other useful controls: "modulo_targets", "non_modulo_markers", "matched_random_non_modulo".
DPE_CAUSAL_INTERVENTION_MARKERS = "all_measured_markers"

# Which candidate to test. By default, the top-ranked component from the whole selected-feature scan.
DPE_CAUSAL_CANDIDATE_RANK = 1


def intervention_rows_from_marker_info(marker_info, mode: str, random_seed: int = 0):
    target_mask = marker_target_mask_from_info(marker_info)
    target_rows = np.flatnonzero(target_mask).tolist()
    non_target_rows = np.flatnonzero(~target_mask).tolist()

    if mode == "all_measured_markers":
        return list(range(len(marker_info)))
    if mode == "modulo_targets":
        return target_rows
    if mode == "non_modulo_markers":
        return non_target_rows
    if mode == "matched_random_non_modulo":
        rng = np.random.default_rng(int(random_seed))
        k = min(len(target_rows), len(non_target_rows))
        if k == 0:
            return []
        return sorted(rng.choice(non_target_rows, size=k, replace=False).tolist())
    raise ValueError(
        "mode must be one of 'all_measured_markers', 'modulo_targets', "
        "'non_modulo_markers', or 'matched_random_non_modulo'."
    )


def flat_intervention_positions_from_index(marker_info, position_index, mode: str, random_seed: int = 0):
    rows = intervention_rows_from_marker_info(marker_info, mode=mode, random_seed=random_seed)
    return flatten_positions_for_rows(position_index, rows=rows).astype(int).tolist()


@torch.no_grad()
def capture_target_feature_with_component_intervention(
    prompt: str,
    *,
    target_sae,
    target_layer: int,
    feature_idx: int,
    component_type: str,
    component_layer: int,
    component_id: int,
    alpha: float,
    position_offsets=DPE_COMPONENT_POSITION_OFFSETS,
    intervention_markers: str = DPE_CAUSAL_INTERVENTION_MARKERS,
    random_seed: int = 0,
):
    """Run one prompt with one component scaled, then measure the target feature at all marker positions."""
    target_layer = int(target_layer)
    component_layer = int(component_layer)
    component_id = int(component_id)

    enc, marker_info, position_index = build_dpe_marker_position_matrix_for_prompt(
        prompt,
        position_offsets=position_offsets,
        skip_first_marker=True,
    )
    flat_measure_positions = position_index["flat_positions"].astype(int).tolist()
    flat_intervention_positions = flat_intervention_positions_from_index(
        marker_info,
        position_index,
        mode=intervention_markers,
        random_seed=random_seed,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    cache = {"target_hidden_flat": None}
    handles = []

    def target_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        idx = torch.tensor(flat_measure_positions, device=x.device, dtype=torch.long)
        cache["target_hidden_flat"] = x[0, idx, :].detach().contiguous()
        return output

    try:
        if component_type == "mlp":
            mlp_module, _ = get_mlp_residual_write_module(model, component_layer)

            def mlp_hook(module, inputs, output):
                if isinstance(output, tuple):
                    x = output[0]
                    rest = output[1:]
                else:
                    x = output
                    rest = None
                if len(flat_intervention_positions) == 0:
                    return output
                idx = torch.tensor(flat_intervention_positions, device=x.device, dtype=torch.long)
                idx = torch.unique(idx[(idx >= 0) & (idx < x.shape[1])])
                if idx.numel() == 0:
                    return output
                x_new = x.clone()
                x_new[0, idx, :] = x_new[0, idx, :] * (1.0 + float(alpha))
                return (x_new,) + rest if rest is not None else x_new

            handles.append(mlp_module.register_forward_hook(mlp_hook))

        elif component_type == "attention_head":
            attn = model.get_submodule(f"model.layers.{component_layer}.self_attn")
            o_proj = attn.o_proj
            num_heads = int(getattr(attn, "num_heads", getattr(model.config, "num_attention_heads")))
            head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))
            if component_id < 0 or component_id >= num_heads:
                raise ValueError(f"Invalid head id {component_id}; layer {component_layer} has {num_heads} heads.")

            def o_proj_pre_hook(module, inputs):
                x = inputs[0]
                if len(flat_intervention_positions) == 0:
                    return None
                idx = torch.tensor(flat_intervention_positions, device=x.device, dtype=torch.long)
                idx = torch.unique(idx[(idx >= 0) & (idx < x.shape[1])])
                if idx.numel() == 0:
                    return None
                x_new = x.clone()
                s = int(component_id) * int(head_dim)
                e = s + int(head_dim)
                x_new[0, idx, s:e] = x_new[0, idx, s:e] * (1.0 + float(alpha))
                return (x_new,)

            handles.append(o_proj.register_forward_pre_hook(o_proj_pre_hook))
        else:
            raise ValueError("component_type must be 'attention_head' or 'mlp'.")

        target_block = model.get_submodule(f"model.layers.{target_layer}")
        handles.append(target_block.register_forward_hook(target_hook))

        _ = get_backbone_model(model)(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        for h in handles:
            h.remove()

    if cache["target_hidden_flat"] is None:
        raise RuntimeError("Failed to capture target hidden states during causal intervention.")

    selected_hidden = make_selected_hidden_from_flat(cache["target_hidden_flat"], position_index)
    target_feature_values = sae_single_feature_values_for_selected_hidden(
        target_sae,
        selected_hidden,
        feature_idx=feature_idx,
    )

    return {
        "marker_info": marker_info,
        "target_mask": marker_target_mask_from_info(marker_info),
        "target_feature_values": np.asarray(target_feature_values, dtype=np.float64),
    }


def summarize_dpe_causal_records(records_df: pd.DataFrame, expected_sign: float):
    rows = []
    for alpha, g in records_df.groupby("alpha"):
        vals = g["target_feature_mean"].to_numpy(dtype=np.float64)
        mask = g["is_modulo10_target"].to_numpy(dtype=bool)
        prompt_ids = g["prompt_idx"].to_numpy(dtype=np.int64)
        fs = summarize_dpe_feature_values(vals, prompt_ids, mask, expected_sign=expected_sign)
        rows.append({
            "alpha": float(alpha),
            "effective_multiplier": 1.0 + float(alpha),
            "n_events": int(len(g)),
            "n_prompts": int(g["prompt_idx"].nunique()),
            **fs,
        })
    out = pd.DataFrame(rows).sort_values("alpha").reset_index(drop=True)
    if (out["alpha"] == 0.0).any():
        base = out.loc[out["alpha"] == 0.0].iloc[0]
        for col in [
            "feature_target_mean", "feature_other_mean", "feature_target_minus_other_mean",
            "feature_promptwise_modulo_z_contrast", "feature_point_biserial_r_target_marker",
            "signed_feature_promptwise_modulo_z_contrast", "signed_feature_point_biserial_r_target_marker",
        ]:
            out[f"delta_{col}_vs_alpha0"] = out[col] - float(base[col])
    return out


def run_dpe_component_causal_test(
    candidate_row,
    *,
    prompts=original_prompts,
    alphas=DPE_CAUSAL_ALPHAS,
    max_prompts=DPE_CAUSAL_MAX_PROMPTS,
    intervention_markers=DPE_CAUSAL_INTERVENTION_MARKERS,
    position_offsets=DPE_COMPONENT_POSITION_OFFSETS,
    verbose_every: int = 10,
):
    """Run alpha sweep for one candidate component row from dpe_component_summary_df."""
    if isinstance(candidate_row, pd.Series):
        row = candidate_row
    else:
        row = pd.Series(candidate_row)

    target_layer = int(row["target_layer"])
    feature_idx = int(row["feature_idx"])
    expected_sign = float(row.get("expected_sign", 1.0))
    component_type = str(row["component_type"])
    component_layer = int(row["component_layer"])
    component_id = int(row.get("component_id", -1))
    direction = str(row.get("direction", ""))
    component_label = _component_label(row)

    if component_layer > target_layer:
        raise ValueError(
            f"Component layer {component_layer} is later than target layer {target_layer}. "
            "Choose component layers <= target layer so the intervention can affect the measured feature."
        )

    prompts_to_run = prompts[:max_prompts] if max_prompts is not None else prompts
    target_sae = load_sae_for_layer(target_layer)

    rows = []
    print("=" * 120)
    print("DPE causal component intervention")
    print("candidate:", component_label)
    print("target layer / feature:", target_layer, feature_idx)
    print("intervention markers:", intervention_markers)
    print("alphas:", alphas)
    print("n prompts:", len(prompts_to_run))
    print("=" * 120)

    for alpha in alphas:
        for prompt_idx, prompt in enumerate(prompts_to_run):
            measured = capture_target_feature_with_component_intervention(
                prompt,
                target_sae=target_sae,
                target_layer=target_layer,
                feature_idx=feature_idx,
                component_type=component_type,
                component_layer=component_layer,
                component_id=component_id,
                alpha=float(alpha),
                position_offsets=position_offsets,
                intervention_markers=intervention_markers,
                random_seed=prompt_idx,
            )
            vals = measured["target_feature_values"]
            mask = measured["target_mask"]
            for i, info in enumerate(measured["marker_info"]):
                rows.append({
                    "alpha": float(alpha),
                    "effective_multiplier": 1.0 + float(alpha),
                    "prompt_idx": int(prompt_idx),
                    "marker_number": int(info["marker_number"]),
                    "completed_examples": int(info["completed_examples"]),
                    "is_modulo10_target": bool(mask[i]),
                    "target_feature_mean": float(vals[i]),
                    "target_layer": int(target_layer),
                    "feature_idx": int(feature_idx),
                    "expected_sign": float(expected_sign),
                    "component_type": component_type,
                    "component_layer": int(component_layer),
                    "component_id": int(component_id),
                    "direction": direction,
                    "component_label": component_label,
                    "intervention_markers": intervention_markers,
                })
            if verbose_every is not None and (prompt_idx + 1) % int(verbose_every) == 0:
                print(f"alpha={alpha}: processed {prompt_idx + 1}/{len(prompts_to_run)} prompts")

    records_df = pd.DataFrame(rows)
    summary_df = summarize_dpe_causal_records(records_df, expected_sign=expected_sign)
    for col, val in {
        "target_layer": target_layer,
        "feature_idx": feature_idx,
        "expected_sign": expected_sign,
        "component_type": component_type,
        "component_layer": component_layer,
        "component_id": component_id,
        "direction": direction,
        "component_label": component_label,
        "intervention_markers": intervention_markers,
    }.items():
        summary_df[col] = val

    safe_label = component_label.replace(" ", "_").replace("-", "_")
    out_dir = os.path.join(OUTPUT_DIR, f"dpe_causal_{safe_label}_targetL{target_layer}_feat{feature_idx}_{intervention_markers}")
    os.makedirs(out_dir, exist_ok=True)
    records_df.to_csv(os.path.join(out_dir, "dpe_component_causal_records.csv"), index=False)
    summary_df.to_csv(os.path.join(out_dir, "dpe_component_causal_summary.csv"), index=False)

    display(summary_df)
    fig = px.line(
        summary_df,
        x="alpha",
        y="signed_feature_promptwise_modulo_z_contrast",
        markers=True,
        title=f"Causal DPE component test: {component_label}<br>target L{target_layer} feature {feature_idx}",
    )
    fig.add_hline(y=0, line_dash="dash", opacity=0.35)
    fig.update_layout(
        width=900,
        height=600,
        template="simple_white",
        font=dict(family="Times New Roman"),
        title=dict(font=dict(size=26)),
        xaxis=dict(title=dict(text="alpha, effective multiplier = 1 + alpha", font=dict(size=22)), tickfont=dict(size=16)),
        yaxis=dict(title=dict(text="Signed DPE feature contrast", font=dict(size=22)), tickfont=dict(size=16)),
    )
    fig.show(config={"toImageButtonOptions": {"format": "png", "filename": "dpe_component_causal", "width": 900, "height": 600, "scale": 4}})

    print("Saved causal results to:", out_dir)
    return summary_df, records_df


if RUN_DPE_COMPONENT_CAUSAL:
    if "dpe_component_summary_df" not in globals() or len(dpe_component_summary_df) == 0:
        raise RuntimeError("Run the component scan first to create dpe_component_summary_df.")
    causal_candidate_row = dpe_component_summary_df.sort_values("dpe_write_score", ascending=False).iloc[int(DPE_CAUSAL_CANDIDATE_RANK) - 1]
    dpe_component_causal_summary_df, dpe_component_causal_records_df = run_dpe_component_causal_test(
        causal_candidate_row,
        prompts=original_prompts,
        alphas=DPE_CAUSAL_ALPHAS,
        max_prompts=DPE_CAUSAL_MAX_PROMPTS,
        intervention_markers=DPE_CAUSAL_INTERVENTION_MARKERS,
        position_offsets=DPE_COMPONENT_POSITION_OFFSETS,
        verbose_every=10,
    )
else:
    print("RUN_DPE_COMPONENT_CAUSAL=False. Set it to True after inspecting dpe_top_component_candidates_df.")
